\# Public Dialogue Analyser (v17)

## Purpose
This notebook analyses 65 UK public dialogue documents on science and technology to identify:

- **Shared concerns or hopes** that recur across different technologies
- **Concerns or benefits that are distinctive to public dialogues on artificial intelligence (AI)** compared to other technologies
- **How public dialogue about AI changes over time**

---

## Corpus
The corpus consists of UK public dialogue reports on science and technology, spanning multiple technologies (including AI) and multiple years. Each document is associated with metadata including technology domain(s) and publication year.

PDFs are ingested directly and converted to paragraph-level text for analysis.

---

## Analytical challenge
A central challenge in computationally analysing public dialogue at scale is that technology-specific language strongly shapes textual similarity. When full paragraphs are compared directly, references to particular technologies (e.g. "AI regulation" versus "nuclear regulation") tend to dominate the semantic space. This causes texts to cluster by technology domain rather than by the underlying nature of the public interest, limiting meaningful cross-technology comparison.

---

## Unit of analysis: decontextualised concern or benefit phrases
To address this challenge, the primary unit of analysis is a **concern or benefit phrase** extracted from a paragraph of dialogue text.

- Paragraphs are segmented from the source documents using length thresholds to exclude boilerplate and very short fragments.
- Each paragraph is processed to extract one or more short phrases that express *substantive public concerns or benefits*, rewritten to minimise explicit references to specific technologies where possible.
- These concern phrases are treated as comparable units across technologies and over time.

---

## Analytical approach (overview)
1. **Text extraction and preprocessing** — PDF documents are converted into paragraph-level text chunks and joined to document metadata.
2. **Concern / benefit extraction** — A large language model extracts concise, decontextualised phrases from each paragraph.
3. **Embedding and clustering** — Phrases are embedded into a semantic vector space and clustered to identify cross-cutting categories.
4. **Comparative and temporal analysis** — AI dialogues are compared against a non-AI baseline to identify distinctive profiles, and AI salience is analysed over time.

---

## Reproducibility and traceability
All figures and summary tables are derived from intermediate datasets that retain links back to the original paragraph-level text. These traceability datasets can be inspected to verify how each concern category and trend relates to the source material.

---

## How to use this notebook
This notebook is intended to be run **top-to-bottom**. All required inputs are loaded near the top, and all figures and tables used in reporting are generated in later sections.

---

## v16 changelog (relative to v15.1)

This release applies a set of audit-driven fixes. It does **not** redesign the methodology. Headline changes:

**Methodologically substantive:**
- **Removed technology-metadata leak in cluster labelling** (Issue 2 in the audit). The labelling prompt no longer tells the LLM the source technology of each exemplar phrase, and no longer flags whether the cluster is cross-cutting. Previous behaviour caused clusters labelled e.g. "AI-driven job displacement" even when every exemplar said "technology displacing employment". Same fix applied to the benefit labeller.
- **Lowered chunking floor from 50→40 words and 100→80 chars.** v15 systematically excluded short, list-formatted paragraphs that dominate recent AI public-attitudes reports (Ada Lovelace, Home Office, BEIS Tracker, ONS). The 40/80 floor recovers ~1,000 substantive participant quotes and short-but-real chunks while still excluding most bibliography fragments and survey-table rows. Sensitivity test the floor at 50 if AI distinctiveness numbers shift materially.
- **Stricter decontextualisation filter** (Issue 6). Word-boundary regex (was substring match, which silently dropped phrases containing `judgment`/`stigma` because of the `gm` substring); expanded term list to include `algorithm`, `automated`, `automation`, `crispr`, `radiation`, `engineered`, `autonomous`, `biometric`, `surveillance`. The token `data` is deliberately *not* filtered because it carries analytic meaning across most technology domains.
- **Cross-cutting threshold consistency** (Issue 7). Cell 30 (concern Section 6 prerequisite) and cell 49 (concern stable-core figure) and cell 88 (benefit stable-core figure) now use **normalised** entropy with threshold 0.5, consistent with the rest of the notebook. v15 silently used raw entropy in those cells, with the same nominal threshold, which classified ~75% of clusters as cross-cutting where the canonical definition classifies ~76% — but the inconsistency made the figures hard to interpret.

**Robustness:**
- **Bumped `max_completion_tokens` from 500 to 1500** for both concern and benefit extraction. v15 had 30 + 25 chunks fail on `BadRequestError: max_tokens reached`. Added a single retry on transient API errors.
- **Fixed `n_init` mismatch** in concern k-sensitivity (cell 104). Baseline used `n_init=10`; sensitivity reruns used `n_init='auto'` (which resolves to 1 for k-means++). Now consistent.
- **New: chunk content-quality diagnostic.** Reports the proportion of kept chunks that look like bibliography fragments (e.g. "et al. (Year)") or survey-table rows (heavy `%` content) so the analyst can see the contamination rate at the chosen floor.

**New diagnostic outputs (do not change methodology):**
- **Document-weighted baseline alongside tech-weighted** for AI distinctiveness (Issue 10 partial). Two CSVs are produced; the analyst can confirm which baseline is being used in the paper.
- **4-window temporal analysis** (2004-2017, 2018-2020, 2021-2023, 2024-2025) alongside year-by-year (Issue 4 partial). Wider windows have larger N and avoid the small-sample entropy artefact.
- **Human validation sample export.** A 250-paragraph stratified-by-technology sample is exported with LLM extractions, ready for hand-coding (Issue 12 partial).
- **Stub for prompt sensitivity analysis** (Issue 5). A clearly-marked scaffold cell documents the analysis that the paper claims to have run. Not actually run — implementing it requires a second extraction pass with alternative prompts.

**Removed dead cells (carried over from v12b for backwards compatibility):**
- Old "load inputs from previous run" zip-upload cell (was used in early development; downstream cells default ARTIFACT_DIR safely).
- Four ARTIFACT_DIR debugging probes (cells 47, 48, 86, 87 in v15).
- Two `print(df.columns)` debug cells.
- One disabled "privacy in benefits" cell.

**Note on what is NOT in scope.** Multi-seed k-means sensitivity is not added (would triple sensitivity runtime). The labelling-leak fix means cluster compositions are unchanged but cluster *labels* will differ from v15; downstream tables that quote labels (the AI-distinctive concern table, etc.) will look different even on identical embeddings. Re-extraction is required because the decontextualisation filter changed.


---

## v17 changelog (relative to v16)

This release is a **cleanup pass** on v16. No methodological changes; existing
findings are preserved. Headline changes:

**Additions:**
- Concern-cluster dendrogram (after framing-lens prevalence). Validates the
  LLM-derived framing-lens scheme against a data-driven hierarchy.
- Benefit-cluster dendrogram (parallel for benefits).
- Human validation export for benefits (mirroring the concern equivalent).
- Markdown header introducing the appendix/supplementary section.

**Removals:**
- Old "Analyse AI-specific concerns/benefits" cells that printed v12-era
  cluster-level "AI is not distinctive" narratives that contradicted the
  paper and were subsumed by the AI-distinctiveness tables.
- Year-over-year stability cells (orphaned; superseded by wider-window
  analysis).
- Privacy keyword regex deep dive (redundant with lens-level analysis).
- Prompt sensitivity scaffold (replaced by the standalone
  prompt_sensitivity_v16.ipynb notebook).

**Code hygiene:**
- Stripped historical v15/v16 annotations from inline comments. Substantive
  current-state comments retained. The change history now lives only in
  this top-level changelog.


---

## v18 changelog (relative to v17)

This release addresses an issue identified in v17 where 20 of 66 documents (mostly recent AI policy reports) were silently truncated to roughly 500 words because the source PDFs lacked double-newline paragraph separators. PyMuPDF returned the entire document text as a single string, which the chunker treated as one giant paragraph and capped at 500 words. The flag `was_truncated=True` was correctly recorded in v17 outputs but not surfaced anywhere in analysis. Affected documents were disproportionately AI policy reports from 2020+, including key policy-relevant reports such as the Ada Lovelace Institute's "Licence to build" briefing, Doteveryone's digital attitudes report, and several ONS surveys.

**Chunker fix**: the chunker now uses two methods.

- **Method 1** (default, unchanged from v17): split on double-newlines (`\n\s*\n`). Used when this produces ≥3 substantive paragraphs and no paragraph exceeds the word cap.
- **Method 2** (new fallback): when Method 1 fails (too few paragraphs or any paragraph would be truncated), discard paragraph structure and split on sentence boundaries instead, repacking sentences into chunks of approximately 300 words.

The decision is made per-document. Documents that chunk well via Method 1 produce identical chunks to v17. Documents that v17 was failing on now produce many more chunks of ~300 words each. Each chunk has a new `chunking_method` column recording which method was used, and the per-document summary CSV now reports method-by-document.

**Other changes**:

- Added `fig.write_image()` to the four radar plot cells so they save PNG copies alongside the existing HTML files (requires the `kaleido` package; falls back gracefully if not installed).
- Added `plt.savefig()` to the two stable-core scatter cells so Figure 1 saves to PNG without manual intervention.
- Added a new "Paper assets" section near the end of the notebook (before the supplementary appendix block). This regenerates Tables 1, 2 and Figures 3, 4 — paper-ready versions of the headline analytical outputs — and writes them to a `paper_assets/` subfolder of `OUTPUT_FOLDER`. The cells are self-contained: they re-load CSVs from `OUTPUT_FOLDER` rather than relying on in-memory state, so they can be re-run individually.

**Expected impact on results**: the chunker fix should add roughly 800-1500 chunks to the corpus, mostly from the 20 affected documents, mostly AI. Existing findings are likely to strengthen rather than change qualitatively, because the affected documents contain substantial content on data, transparency, and oversight themes — exactly the informational autonomy themes that drive the AI distinctiveness finding.


---

## v19 changelog (relative to v18)

This release walks back a methodological choice in v18 that was more aggressive than intended. v18 introduced a two-method chunker (paragraph splitting with sentence-fallback). The trigger for switching to sentence-fallback was "any single paragraph in the document exceeds the truncation cap" — which fired for 58 of 66 documents because most substantial reports contain at least one long paragraph somewhere. The effect was that the analytic unit shifted from "paragraph as the author wrote it" to "approximately 300-word window of contiguous sentences" for nearly the entire corpus.

That was a bigger conceptual shift than the issue warranted. v17's truncation problem only meaningfully affected documents whose PDFs lacked any paragraph break encoding — perhaps 5-10 documents in the corpus. The other ~55 documents were chunking correctly via paragraph splitting; they just happened to have one or two long paragraphs that would trigger v18's overly broad trigger condition.

**v19 chunker — three cases per document**:

- **Paragraph-only segmentation (most documents)**: paragraph splitting produces ≥3 substantive paragraphs and no paragraph exceeds the truncation cap. All chunks are paragraphs as the author wrote them. Identical to v17 behaviour.
- **Paragraph segmentation with internal sentence-splitting (some documents)**: paragraph splitting produces enough substantive paragraphs but at least one is oversized. The well-sized paragraphs are kept as-is; only oversized paragraphs are sentence-split into sub-chunks of ~300 words. The author's paragraph boundary decisions are preserved everywhere they exist; only individual long paragraphs get further subdivision.
- **Full sentence-fallback (few documents)**: paragraph splitting produces fewer than 3 substantive paragraphs (PyMuPDF cannot extract paragraph breaks). The entire document is sentence-split. This recovers content from PDFs that lack paragraph break encoding — the original problem the v18 fix was designed to solve.

Each chunk records which method produced *it* via `chunking_method`: `paragraph`, `sentence_split` (oversized paragraph split internally), or `sentence_fallback` (case-3 document-level fallback).

**Expected impact**: most documents return to v17 paragraph segmentation. The truncation problem is still solved (oversized paragraphs are split, not truncated). The original "PDFs with no paragraph breaks" problem is still solved (case 3). The total chunk count should sit between v17 and v18; the *segmentation* of most chunks reverts to v17 behaviour.

**Methodology paragraph for the paper**: "We segmented PDFs into paragraphs as written by the author. Paragraphs longer than 500 words were further split at sentence boundaries to avoid information loss. Documents where PyMuPDF could not extract paragraph breaks at all (n=X of 66) were segmented at sentence boundaries throughout. The chunking method used for each chunk is recorded in the output."


# PART I: Concern space (v10 pipeline, unchanged)

This part reproduces the original v10 concern pipeline.

---


---
# SECTION 0: Setup and configuration
---

In [ ]:
# @title Install analysis dependencies
!pip install -q PyMuPDF openai scikit-learn umap-learn plotly kaleido openpyxl tqdm scipy

In [ ]:
# @title Import libraries and define configuration
import os
import json
import time
import re
import random
from pathlib import Path
from collections import Counter
from datetime import datetime
from typing import List, Dict
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import numpy as np
import fitz  # PyMuPDF
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.stats import entropy
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from tqdm.notebook import tqdm
from openai import OpenAI
from IPython.display import display, HTML

# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Model Configuration
EMBEDDING_MODEL = "text-embedding-3-large"
LLM_MODEL = "gpt-4o-mini"

# Clustering - now 75 clusters for concern phrases (smaller units than paragraphs)
N_CONCERN_CLUSTERS = 75
SOFT_MEMBERSHIP_THRESHOLD = 0.3

# Chunking
MIN_CHUNK_WORDS = 40
MAX_CHUNK_WORDS = 500
MIN_CHUNK_CHARS = 80

# When paragraph-level splitting (the default) fails to produce reasonable
# chunks (because the PDF lacks double-newline paragraph separators),
# fall back to sentence-level splitting and repack into chunks of about
# this many words. v18: addresses an issue identified in v17 where 20
# documents (mostly AI policy reports from 2020+) were silently truncated
# because their full text was treated as a single 500-word paragraph.
SENTENCE_FALLBACK_TARGET_WORDS = 300
SENTENCE_FALLBACK_MIN_PARAGRAPHS = 3  # if Method 1 yields fewer paragraphs, fall back

# Processing
EMBEDDING_BATCH_SIZE = 100
EXTRACTION_BATCH_SIZE = 10  # For concern extraction

# Paths
PDF_FOLDER = Path("/content/pdfs")
OUTPUT_FOLDER = Path("/content/outputs")
CHECKPOINT_FOLDER = Path("/content/checkpoints")

for folder in [PDF_FOLDER, OUTPUT_FOLDER, CHECKPOINT_FOLDER]:
    folder.mkdir(exist_ok=True)

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 150

import warnings
warnings.filterwarnings('ignore')

print("Configuration loaded")
print(f"  Embedding model: {EMBEDDING_MODEL}")
print(f"  Target clusters: {N_CONCERN_CLUSTERS}")
print(f"  LLM for extraction: {LLM_MODEL}")

In [ ]:
# @title Load dialogue_utils — shared helper functions
# All helper functions (show_status, show_complete, show_warning,
# save_checkpoint, load_checkpoint, extract_chunks_from_pdf, extract_phrases,
# label_cluster, normalized_entropy, run_sensitivity, etc.) are imported from
# dialogue_utils.py.  The module must be on sys.path (it is fetched in the
# cell below).

try:
    import dialogue_utils as du
    from dialogue_utils import (
        show_status, show_complete, show_warning,
        save_checkpoint, load_checkpoint,
        CROSSCUTTING_ENTROPY_THRESHOLD,
        EXTRACTION_PROMPT, BENEFIT_EXTRACTION_PROMPT,
        ExtractionResult, extract_phrases, validate_extraction_cache,
        write_extraction_diagnostics,
        filter_missing_source_text, get_embeddings_batch,
        label_cluster, pretty_label, clusters_to_labels,
        clusters_to_lenses, html_escape,
        normalized_entropy, hhi, topk_share, parse_year, tokenize,
        is_privacy_text, entropy_by_year, ai_fingerprint_over_crosscut,
        run_sensitivity,
        vocabulary_frequency_diagnostic, generate_validation_summary,
        extract_chunks_from_pdf, reset_chunk_stats, get_chunk_stats,
        MIN_CHUNK_WORDS, MIN_CHUNK_CHARS, MAX_CHUNK_WORDS,
        SENTENCE_FALLBACK_TARGET_WORDS, SENTENCE_FALLBACK_MIN_PARAGRAPHS,
    )
    print("dialogue_utils imported successfully")
except ImportError as _e:
    print(f"dialogue_utils not found: {_e}")
    print("Fetch it with: !wget https://raw.githubusercontent.com/mlatcl/pub-dialogue/main/dialogue_utils.py")
    raise

In [ ]:
# @title Configure API access
import os as _os

api_key = None

# 1. Try Colab secrets (when running in Google Colab)
try:
    from google.colab import userdata as _userdata
    api_key = _userdata.get('OPENAI_API_KEY')
    if api_key:
        print("API key loaded from Colab secrets")
except Exception:
    pass

# 2. Try environment variable (when running locally)
if not api_key:
    api_key = _os.environ.get("OPENAI_API_KEY")
    if api_key:
        print("API key loaded from OPENAI_API_KEY environment variable")

# 3. Interactive prompt fallback
if not api_key:
    from getpass import getpass as _getpass
    api_key = _getpass("Enter OpenAI API key: ")
    print("API key entered manually")

client = OpenAI(api_key=api_key)

try:
    client.models.list()
    print("API connection verified")
except Exception as e:
    print(f"API error: {e}")

---
# SECTION 1: Data ingestion and preprocessing
---

In [ ]:
# @title Upload source PDF documents
show_status("Preparing PDF upload...")

from google.colab import files

print("Upload your PDF files:")
uploaded = files.upload()

pdf_files = []
for filename, content in uploaded.items():
    if filename.endswith('.pdf'):
        filepath = PDF_FOLDER / filename
        filepath.write_bytes(content)
        pdf_files.append(filepath)

show_complete(f"Uploaded {len(pdf_files)} PDF files")

In [ ]:
# @title Upload document metadata
show_status("Upload metadata file...")

print("Upload Excel file with columns: filename, technology, year")
uploaded = files.upload()

metadata_df = None
for fn, content in uploaded.items():
    if fn.endswith(('.xlsx', '.xls')):
        path = OUTPUT_FOLDER / fn
        path.write_bytes(content)
        metadata_df = pd.read_excel(path)
        break

if metadata_df is None:
    raise ValueError("No Excel file uploaded!")

required = ['filename', 'technology', 'year']
missing = [c for c in required if c not in metadata_df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

metadata_lookup = metadata_df.set_index('filename').to_dict('index')

print(f"\nTechnology Distribution:")
print(metadata_df['technology'].value_counts().to_string())
print(f"\nYear Range: {metadata_df['year'].min()} - {metadata_df['year'].max()}")

show_complete(f"Metadata loaded for {len(metadata_df)} documents")

In [ ]:
# @title Extract paragraph-level text from PDFs (v19 three-case hybrid chunker)
# Helper functions (_split_into_sentences, _repack_sentences_into_chunks,
# _paragraph_split, extract_chunks_from_pdf) are provided by dialogue_utils.
# The three-case strategy:
#   Case 1 — paragraph-only (most documents)
#   Case 2 — paragraphs + internal sentence-split of oversized paragraphs
#   Case 3 — full sentence-fallback (PDFs lacking paragraph breaks)

show_status("Extracting paragraph chunks...")

reset_chunk_stats()

all_chunks = []
for pdf_path in tqdm(list(PDF_FOLDER.glob("*.pdf")), desc="Processing PDFs"):
    if pdf_path.name not in metadata_lookup:
        raise ValueError(f"No metadata found for PDF: {pdf_path.name}")
    metadata = metadata_lookup[pdf_path.name]
    chunks = extract_chunks_from_pdf(pdf_path, metadata)
    all_chunks.extend(chunks)

chunks_df = pd.DataFrame(all_chunks)
chunks_df["chunk_id"] = [f"chunk_{i}" for i in range(len(chunks_df))]

_chunk_stats = get_chunk_stats()

print(f"\nExtraction Summary:")
print(f"  Documents:                                     {chunks_df['source_file'].nunique()}")
print(f"  Documents w/ pure paragraph segmentation:      {_chunk_stats['documents_paragraph_only']}")
print(f"  Documents w/ paragraphs + some sentence-split: {_chunk_stats['documents_paragraph_with_split']}")
print(f"  Documents w/ full sentence-fallback:           {_chunk_stats['documents_sentence_fallback']}")
print(f"  Oversized paragraphs split internally:         {_chunk_stats['oversized_paragraphs_split']}")
print(f"  Chunks produced by sentence-split:             {_chunk_stats['chunks_from_sentence_split']}")
print(f"  Chunks produced by full sentence-fallback:     {_chunk_stats['chunks_from_sentence_fallback']}")
print(f"  Total chunks (paragraph-derived):              {(chunks_df['chunking_method']=='paragraph').sum()}")
print(f"  Paragraphs seen (raw):                         {_chunk_stats['paragraphs_seen']}")
print(f"  Below {MIN_CHUNK_WORDS}-word floor:                          {_chunk_stats['paragraphs_below_word_floor']}")
print(f"  Below {MIN_CHUNK_CHARS}-char floor:                          {_chunk_stats['paragraphs_below_char_floor']}")
print(f"  Truncated (safety net):                        {_chunk_stats['paragraphs_truncated']}")
print(f"  Kept after filter:                             {_chunk_stats['paragraphs_kept']}")
print(f"  Words per kept chunk (mean):                   {chunks_df['word_count'].mean():.0f}")

# Per-document summary
doc_summary = (chunks_df.groupby("source_file")
               .agg(paragraphs_kept=("chunk_id", "size"),
                    methods_used=("chunking_method", lambda s: "+".join(sorted(s.unique()))),
                    truncated_chunks=("was_truncated", "sum"))
               .reset_index()
               .sort_values("paragraphs_kept"))
doc_summary.to_csv(OUTPUT_FOLDER / "paragraph_chunks_per_document.csv", index=False)
print(f"\nDocuments using full sentence-fallback (PDF lacks paragraph breaks):")
fallback_docs = doc_summary[doc_summary["methods_used"] == "sentence_fallback"]
print(fallback_docs.to_string(index=False) if len(fallback_docs) else "  (none)")

chunks_df.to_csv(OUTPUT_FOLDER / "paragraph_chunks.csv", index=False)
show_complete(f"Extracted {len(chunks_df)} chunks from {chunks_df['source_file'].nunique()} documents")

TECHNOLOGY_CATEGORIES = sorted(metadata_df["technology"].dropna().unique().tolist())

# Sanity check: technology labels remain canonical
observed = set(chunks_df["technology_meta"].unique().tolist())
expected = set(TECHNOLOGY_CATEGORIES)
if observed != expected:
    print("\nWARNING: Technology labels in extracted chunks differ from metadata!")
    print("Observed-only:", sorted(observed - expected))
    print("Metadata-only:", sorted(expected - observed))

In [ ]:
# @title Summarise paragraph-level data quality

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Chunks by technology
tech_counts = chunks_df['technology_meta'].value_counts()
axes[0, 0].barh(tech_counts.index, tech_counts.values, color='steelblue')
axes[0, 0].set_xlabel('Number of Paragraphs')
axes[0, 0].set_title('Paragraphs by Technology')
axes[0, 0].invert_yaxis()

# Chunks by year
year_counts = chunks_df.groupby('year').size()
axes[0, 1].bar(year_counts.index.astype(str), year_counts.values, color='steelblue')
axes[0, 1].set_xlabel('Year')
axes[0, 1].set_ylabel('Paragraphs')
axes[0, 1].set_title('Paragraphs by Year')
axes[0, 1].tick_params(axis='x', rotation=45)

# Word count distribution
axes[1, 0].hist(chunks_df['word_count'], bins=30, color='steelblue', edgecolor='white')
axes[1, 0].set_xlabel('Word Count')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Paragraph Length Distribution')

# Chunks per document
doc_chunks = chunks_df.groupby('source_file').size().sort_values(ascending=False)
axes[1, 1].bar(range(len(doc_chunks)), doc_chunks.values, color='steelblue')
axes[1, 1].set_xlabel('Document Index')
axes[1, 1].set_ylabel('Paragraphs')
axes[1, 1].set_title('Paragraphs per Document')

plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "data_quality_overview.png", dpi=150)
plt.show()

In [ ]:
# @title (v16) Chunk content-quality diagnostic
# Reports the proportion of kept chunks that look like bibliography fragments
# or survey-table rows, so the analyst can see the contamination rate at the
# chosen MIN_CHUNK_WORDS floor.

show_status("Running chunk content-quality diagnostic...")

import re

# Heuristics. Kept deliberately simple — these are diagnostics, not filters.
_BIB_PATTERN = re.compile(r"\b(et al\.?|doi:|http(s)?://|pp?\.\s*\d+|vol\.|isbn|issn)\b", re.IGNORECASE)
_TABLE_PATTERN = re.compile(r"%")  # any percent-sign occurrence
_CITATION_YEAR = re.compile(r"\([12][0-9]{3}\)")  # (1999) / (2024) etc.

def _looks_like_bibliography(text: str) -> bool:
    if not isinstance(text, str):
        return False
    return bool(_BIB_PATTERN.search(text)) and bool(_CITATION_YEAR.search(text))

def _looks_like_table_row(text: str) -> bool:
    if not isinstance(text, str):
        return False
    pct_count = len(_TABLE_PATTERN.findall(text))
    # Heuristic: 2+ percent signs in a chunk strongly suggests a survey table row
    return pct_count >= 2

chunks_df["likely_bibliography"] = chunks_df["text"].apply(_looks_like_bibliography)
chunks_df["likely_table_row"] = chunks_df["text"].apply(_looks_like_table_row)

n_total = len(chunks_df)
n_bib = int(chunks_df["likely_bibliography"].sum())
n_table = int(chunks_df["likely_table_row"].sum())

print(f"Chunk content quality (n = {n_total}):")
print(f"  Likely bibliography fragment: {n_bib}  ({n_bib/n_total*100:.1f}%)")
print(f"  Likely survey-table row:      {n_table}  ({n_table/n_total*100:.1f}%)")
print()
print("These chunks are kept for analysis (the diagnostic does not filter).")
print("If the contamination rate is high, consider raising MIN_CHUNK_WORDS.")
print(f"Current floor: {MIN_CHUNK_WORDS} words / {MIN_CHUNK_CHARS} chars.")

# Write the flagged chunks for manual inspection
flagged = chunks_df[chunks_df["likely_bibliography"] | chunks_df["likely_table_row"]][
    ["chunk_id", "source_file", "word_count", "likely_bibliography", "likely_table_row", "text"]
]
flagged.to_csv(OUTPUT_FOLDER / "chunk_quality_flagged.csv", index=False)
show_complete(f"Wrote {len(flagged)} flagged chunks to chunk_quality_flagged.csv (for inspection)")


---
# SECTION 2: Concern extraction
---

This section extracts decontextualised public concern phrases from paragraph-level dialogue text. These concern phrases form the primary analytic units for all subsequent embedding, clustering, and comparative analysis.


In [ ]:
# @title Extract decontextualised concern phrases

from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

show_status("Extracting decontextualised concern phrases...")

EXTRACTION_PROMPT = """Extract the core public concerns from this paragraph from a UK public dialogue report.

CRITICAL RULES:
1. Remove ALL technology-specific references (AI, nuclear, genetic, nano, etc.)
2. Extract the underlying concern that could apply to ANY technology
3. Keep phrases concise (3-10 words each)
4. Focus on what people are worried about, not factual statements

EXAMPLES:
- "People worried about AI making unfair decisions" → "unfair automated decisions"
- "Concerns about nuclear waste storage" → "long-term waste storage safety"
- "Distrust of government handling of genetic data" → "distrust of government data handling"
- "Fear that robots will take jobs" → "technology displacing employment"

Return 1-3 concern phrases, one per line. No bullets, no numbering.
If the paragraph contains no clear public concern, return "NO_CONCERN".

Paragraph:
{text}"""

_concern_extraction_failures = []  # list of (chunk_id, reason)
_concern_extraction_failures_lock = threading.Lock()

def _record_failure(chunk_id, reason):
    with _concern_extraction_failures_lock:
        _concern_extraction_failures.append((chunk_id, reason))

_TECH_TERM_PATTERN = re.compile(
    r"\b("
    r"ai|artificial intelligence|"
    r"nuclear|"
    r"genetic|genome|gene|crispr|embryo|stem cell|"
    r"nano|"
    r"robot|drone|autonomous|"
    r"quantum|"
    r"gm|"
    r"algorithm|algorithmic|automated|automation|"
    r"radiation|engineered|biometric|surveillance"
    r")\b",
    re.IGNORECASE,
)

def _phrase_contains_tech_term(phrase: str) -> bool:
    return bool(_TECH_TERM_PATTERN.search(phrase or ""))

def extract_concerns_from_paragraph(row_tuple):
    """Extract decontextualised concerns from a single paragraph.

    temperature=1). Logs failures instead of silently returning [].
    'max_tokens reached' BadRequestError on ~0.7% of chunks). One retry on
    transient API errors. Word-boundary regex for the post-extraction
    technology-term filter (was substring match).
    """
    idx, row = row_tuple
    chunk_id = row['chunk_id']

    def _call():
        return client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": "Extract public concerns. Be concise. Remove technology-specific language."},
                {"role": "user", "content": EXTRACTION_PROMPT.format(text=row['text'][:2000])}
            ],
            max_completion_tokens=1500,
        )

    try:
        try:
            response = _call()
        except Exception:
            time.sleep(1.0)
            response = _call()  # one retry
        content = response.choices[0].message.content
        if content is None:
            _record_failure(chunk_id, "empty_response")
            return chunk_id, []
        content = content.strip()

        if content == "NO_CONCERN" or not content:
            return chunk_id, []

        concerns = [c.strip() for c in content.split('\n') if c.strip() and c.strip() != "NO_CONCERN"]
        filtered = [c for c in concerns if not _phrase_contains_tech_term(c)]
        return chunk_id, filtered
    except Exception as e:
        _record_failure(chunk_id, f"exception: {type(e).__name__}: {e}")
        return chunk_id, []

# Check for cached extractions
concerns_cache_file = CHECKPOINT_FOLDER / "extracted_concerns.json"

if concerns_cache_file.exists():
    print("Found cached concern extractions...")
    with open(concerns_cache_file) as f:
        cached_concerns = json.load(f)

    cached_ids = set(cached_concerns.keys())
    current_ids = set(chunks_df['chunk_id'].tolist())

    if cached_ids == current_ids:
        print(f"Using cached extractions for {len(cached_concerns)} paragraphs")
        all_concerns = cached_concerns
    else:
        print("Cache mismatch - will re-extract")
        all_concerns = None
else:
    all_concerns = None

if all_concerns is None:
    all_concerns = {}

    # Build lookup for metadata
    chunk_metadata = chunks_df.set_index('chunk_id')[['technology', 'year', 'source_file']].to_dict('index')

    # Parallel extraction with 10 workers
    MAX_WORKERS = 10
    rows = list(chunks_df.iterrows())

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(extract_concerns_from_paragraph, row): row[1]['chunk_id']
                   for row in rows}

        for future in tqdm(as_completed(futures), total=len(futures), desc="Extracting concerns"):
            chunk_id, concerns = future.result()
            meta = chunk_metadata[chunk_id]
            all_concerns[chunk_id] = {
                'concerns': concerns,
                'technology': meta['technology'],
                'year': int(meta['year']) if pd.notna(meta['year']) else None,
                'source_file': meta['source_file']
            }

    # Cache the results
    with open(concerns_cache_file, 'w') as f:
        json.dump(all_concerns, f, indent=2)

    show_complete(f"Extracted concerns from {len(all_concerns)} paragraphs")

if _concern_extraction_failures:
    failure_log_path = OUTPUT_FOLDER / "concern_extraction_failures.csv"
    pd.DataFrame(_concern_extraction_failures, columns=["chunk_id", "reason"]).to_csv(failure_log_path, index=False)
    failure_rate = len(_concern_extraction_failures) / max(1, len(chunks_df))
    show_warning(
        f"Concern extraction had {len(_concern_extraction_failures)} failures "
        f"({failure_rate:.1%} of chunks). See {failure_log_path}."
    )
    if failure_rate > 0.05:
        raise RuntimeError(
            f"Concern extraction failure rate ({failure_rate:.1%}) exceeds 5% threshold. "
            f"Inspect {failure_log_path} before proceeding."
        )

# Flatten to individual concern rows
concern_rows = []
for chunk_id, data in all_concerns.items():
    for concern in data['concerns']:
        concern_rows.append({
            'chunk_id': chunk_id,
            'concern': concern,
            'technology': data['technology'],
            'year': data['year'],
            'source_file': data['source_file']
        })

concerns_df = pd.DataFrame(concern_rows)
concerns_df['concern_id'] = [f"concern_{i}" for i in range(len(concerns_df))]

print(f"\nExtraction Summary:")
print(f"  Paragraphs processed: {len(all_concerns)}")
print(f"  Total concern phrases: {len(concerns_df)}")
print(f"  Avg concerns per paragraph: {len(concerns_df) / max(1, len(all_concerns)):.2f}")
print(f"  Paragraphs with no concerns: {sum(1 for d in all_concerns.values() if len(d['concerns']) == 0)}")

concerns_df.to_csv(OUTPUT_FOLDER / "extracted_concerns.csv", index=False)


In [ ]:
# Make sure concerns_df knows the technology of each paragraph
concerns_df = concerns_df.merge(
    chunks_df[["chunk_id", "technology_meta"]],
    on="chunk_id",
    how="left"
)


In [ ]:
# @title Inspect sample concern extractions

print("Sample extracted concerns by technology:")
print("(Checking that technology-specific language has been removed)\n")

for tech in concerns_df['technology_meta'].unique()[:6]:
    tech_concerns = concerns_df[concerns_df['technology_meta'] == tech]['concern'].head(8).tolist()
    print(f"\n{tech}:")
    for c in tech_concerns:
        print(f"  • {c}")

---
# SECTION 3: Embedding and clustering
---

In [ ]:
# @title Generate semantic embeddings for concern phrases

def get_embeddings_batch(texts, model=EMBEDDING_MODEL):
    response = client.embeddings.create(input=texts, model=model)
    return np.array([item.embedding for item in response.data])

embeddings_file = CHECKPOINT_FOLDER / "concern_embeddings.npy"
concern_ids_file = CHECKPOINT_FOLDER / "concern_ids.json"

if embeddings_file.exists() and concern_ids_file.exists():
    print("Found cached embeddings...")
    embeddings = np.load(embeddings_file)
    with open(concern_ids_file) as f:
        cached_concern_ids = json.load(f)

    if cached_concern_ids == concerns_df['concern_id'].tolist():
        print(f"Using cached embeddings: {embeddings.shape}")
    else:
        print("Cache mismatch - regenerating")
        embeddings = None
else:
    embeddings = None

if embeddings is None:
    show_status(f"Generating embeddings for {len(concerns_df)} concern phrases...")

    texts = concerns_df['concern'].tolist()
    all_embeddings = []
    failed_batch_starts = []  # track failed batches

    for i in tqdm(range(0, len(texts), EMBEDDING_BATCH_SIZE), desc="Embedding"):
        batch = texts[i:i+EMBEDDING_BATCH_SIZE]
        try:
            batch_embeddings = get_embeddings_batch(batch)
            all_embeddings.append(batch_embeddings)
        except Exception as e:
            print(f"Error on batch {i}: {e}")
            failed_batch_starts.append(i)
        time.sleep(0.1)

    if failed_batch_starts:
        raise RuntimeError(
            f"{len(failed_batch_starts)} embedding batch(es) failed: "
            f"starting indices {failed_batch_starts}. "
            f"Re-run this cell after investigating; the cache will be regenerated."
        )

    embeddings = np.vstack(all_embeddings)
    np.save(embeddings_file, embeddings)
    with open(concern_ids_file, 'w') as f:
        json.dump(concerns_df['concern_id'].tolist(), f)

    show_complete(f"Generated embeddings: {embeddings.shape}")

assert embeddings.shape[0] == len(concerns_df), (
    f"Embeddings count ({embeddings.shape[0]}) does not match concerns_df rows ({len(concerns_df)}). "
    f"Delete {embeddings_file} and re-run."
)

print(f"Embedding dimensions: {embeddings.shape[1]}")


In [ ]:
# @title Cluster concern phrase embeddings

show_status(f"Clustering into {N_CONCERN_CLUSTERS} concern groups...")

embeddings_normalized = normalize(embeddings)

kmeans = KMeans(
    n_clusters=N_CONCERN_CLUSTERS,
    random_state=RANDOM_SEED,
    n_init=10,
    max_iter=300
)
concern_cluster_assignments = kmeans.fit_predict(embeddings_normalized)

centroids = kmeans.cluster_centers_
centroids_normalized = normalize(centroids)

assert len(concern_cluster_assignments) == len(concerns_df), (
    f"Cluster assignment length ({len(concern_cluster_assignments)}) does not match "
    f"concerns_df rows ({len(concerns_df)})."
)

# Add cluster assignment to concerns dataframe
concerns_df['cluster_id'] = concern_cluster_assignments

# SOFT MEMBERSHIP: cosine similarity to each centroid
show_status("Computing soft membership scores...")
soft_membership = cosine_similarity(embeddings_normalized, centroids_normalized)

print(f"\nClustering Results:")
print(f"  Clusters: {N_CONCERN_CLUSTERS}")
print(f"  Concern phrases: {len(concerns_df)}")
print(f"  Soft membership matrix: {soft_membership.shape}")

cluster_sizes = np.bincount(concern_cluster_assignments)
print(f"  Cluster sizes: min={cluster_sizes.min()}, max={cluster_sizes.max()}, median={np.median(cluster_sizes):.0f}")

np.save(CHECKPOINT_FOLDER / "soft_membership.npy", soft_membership)
np.save(CHECKPOINT_FOLDER / "cluster_centroids.npy", centroids_normalized)

show_complete("Clustering complete")


In [ ]:
# @title Characterise clusters by technology distribution

show_status("Analysing cluster composition...")

# For each cluster, compute entropy of technology distribution
cluster_entropy = {}
cluster_tech_dist = {}

technologies = concerns_df['technology_meta'].unique()
n_techs = len(technologies)

for cluster_id in range(N_CONCERN_CLUSTERS):
    cluster_mask = concerns_df['cluster_id'] == cluster_id
    cluster_data = concerns_df[cluster_mask]

    if len(cluster_data) == 0:
        cluster_entropy[cluster_id] = 0
        cluster_tech_dist[cluster_id] = {}
        continue

    # Technology distribution
    tech_counts = cluster_data['technology_meta'].value_counts()
    tech_probs = tech_counts / tech_counts.sum()

    # Entropy (higher = more cross-cutting)
    cluster_entropy[cluster_id] = entropy(tech_probs)
    cluster_tech_dist[cluster_id] = tech_counts.to_dict()

# Normalize entropy to 0-1 scale (max entropy = log(n_techs))
max_entropy = np.log(n_techs)
cluster_entropy_norm = {k: v / max_entropy for k, v in cluster_entropy.items()}

# Classify clusters
CROSS_CUTTING_THRESHOLD = 0.5  # Entropy > 50% of max = cross-cutting

cross_cutting_clusters = [k for k, v in cluster_entropy_norm.items() if v >= CROSS_CUTTING_THRESHOLD]
tech_specific_clusters = [k for k, v in cluster_entropy_norm.items() if v < CROSS_CUTTING_THRESHOLD]

print(f"\nCluster Classification:")
print(f"  Cross-cutting clusters (entropy >= {CROSS_CUTTING_THRESHOLD}): {len(cross_cutting_clusters)}")
print(f"  Technology-specific clusters: {len(tech_specific_clusters)}")
print(f"  Ratio: {len(cross_cutting_clusters)/N_CONCERN_CLUSTERS*100:.1f}% cross-cutting")

# Visualise entropy distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Histogram of entropy
axes[0].hist(list(cluster_entropy_norm.values()), bins=20, color='steelblue', edgecolor='white')
axes[0].axvline(CROSS_CUTTING_THRESHOLD, color='red', linestyle='--', label=f'Threshold ({CROSS_CUTTING_THRESHOLD})')
axes[0].set_xlabel('Normalized Entropy')
axes[0].set_ylabel('Number of Clusters')
axes[0].set_title('Cluster Entropy Distribution\n(Higher = More Cross-Cutting)')
axes[0].legend()

# Entropy vs cluster size
sizes = [cluster_sizes[i] for i in range(N_CONCERN_CLUSTERS)]
entropies = [cluster_entropy_norm[i] for i in range(N_CONCERN_CLUSTERS)]
colors = ['green' if e >= CROSS_CUTTING_THRESHOLD else 'orange' for e in entropies]
axes[1].scatter(sizes, entropies, c=colors, alpha=0.6)
axes[1].axhline(CROSS_CUTTING_THRESHOLD, color='red', linestyle='--')
axes[1].set_xlabel('Cluster Size')
axes[1].set_ylabel('Normalized Entropy')
axes[1].set_title('Cross-Cutting (green) vs Technology-Specific (orange)')

plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "cluster_entropy_analysis.png", dpi=150)
plt.show()

show_complete("Cluster composition analysis complete")

In [ ]:
# --- Define cross-cutting helpers for Section 6 (missing in v10-2) ---

# Use the same threshold already used in Section 3
CROSS_CUTTING_ENTROPY_THRESHOLD = CROSS_CUTTING_THRESHOLD  # already 0.5 in the notebook

# IDs of cross-cutting clusters
cross_cutting_cluster_ids = set(cross_cutting_clusters)

# Optional: map cluster_id -> human label if we have GPT labels (summary_df)
label_map = {}
if "summary_df" in globals() and {"cluster_id", "label"}.issubset(summary_df.columns):
    label_map = dict(zip(summary_df["cluster_id"], summary_df["label"]))

# Provide the dict Section 6.2 expects
cross_cutting_labels = {cid: label_map.get(cid, f"Cluster {cid}") for cid in cross_cutting_cluster_ids}


---
# SECTION 4: Cluster interpretation and labelling
---

In [ ]:
# @title Extract exemplar concern phrases per cluster

show_status("Extracting exemplar concerns...")

N_EXEMPLARS = 8
cluster_exemplars = {}

for cluster_id in range(N_CONCERN_CLUSTERS):
    cluster_mask = concerns_df['cluster_id'] == cluster_id
    cluster_concerns = concerns_df[cluster_mask]
    cluster_embs = embeddings_normalized[cluster_mask]

    if len(cluster_concerns) == 0:
        continue

    centroid = centroids_normalized[cluster_id]
    similarities = cosine_similarity(cluster_embs, centroid.reshape(1, -1)).flatten()
    top_indices = np.argsort(similarities)[-N_EXEMPLARS:][::-1]

    exemplars = []
    for idx in top_indices:
        row = cluster_concerns.iloc[idx]
        exemplars.append({
            'concern': row['concern'],
            'technology': row['technology'],
            'year': int(row['year']) if pd.notna(row['year']) else None,
            'similarity': float(similarities[idx])
        })

    # Cluster metadata
    tech_dist = cluster_concerns['technology'].value_counts().head(3).to_dict()

    cluster_exemplars[cluster_id] = {
        'size': len(cluster_concerns),
        'entropy': normalized_entropy[cluster_id],
        'is_cross_cutting': cluster_id in cross_cutting_clusters,
        'top_technologies': tech_dist,
        'exemplars': exemplars
    }

with open(OUTPUT_FOLDER / "cluster_exemplars.json", 'w') as f:
    json.dump(cluster_exemplars, f, indent=2, default=str)

show_complete(f"Extracted exemplars for {len(cluster_exemplars)} clusters")

In [ ]:
# @title Assign descriptive labels to clusters

MIN_CLUSTER_LABEL_SUCCESS_RATE = 0.90

def label_cluster(cluster_id, exemplars, is_cross_cutting):
    exemplar_texts = "\n".join([f"- {ex['concern']}" for ex in exemplars[:8]])

    prompt = f"""Analyze this cluster of public concerns from UK dialogue reports.

Concern phrases in this cluster:
{exemplar_texts}

Provide:
1. SHORT LABEL (3-6 words) capturing the core concern theme.
   Use neutral, generic language; do NOT prefix the label with a specific
   technology (e.g. write "Job displacement", not "AI-driven job displacement").
2. DESCRIPTION (1-2 sentences)
3. KEY TERMS (3-5 representative words/phrases)

Return JSON only:
{{"label": "...", "description": "...", "key_terms": ["...", "..."]}}"""

    try:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": "Expert qualitative researcher. Return only valid JSON."},
                {"role": "user", "content": prompt}
            ],
            max_completion_tokens=1000,
        )

        content = response.choices[0].message.content
        if content is None or not content.strip():
            return {'label': f'Cluster {cluster_id}', 'description': '', 'key_terms': [], 'success': False}

        content = content.strip()
        if '```' in content:
            parts = content.split('```')
            for part in parts:
                if part.startswith('json'):
                    content = part[4:].strip()
                    break
                elif part.strip().startswith('{'):
                    content = part.strip()
                    break

        result = json.loads(content)
        result['success'] = True
        return result
    except Exception as e:
        return {'label': f'Cluster {cluster_id}', 'description': '', 'key_terms': [], 'success': False, 'error': str(e)}

# Check for cached labels
labels_file = OUTPUT_FOLDER / "cluster_labels.json"

if labels_file.exists():
    print("Found cached cluster labels...")
    with open(labels_file) as f:
        cluster_labels_dict = json.load(f)

    # Verify we have labels for all clusters
    if len(cluster_labels_dict) == N_CONCERN_CLUSTERS:
        print(f"Using cached labels for {len(cluster_labels_dict)} clusters")
    else:
        print("Cache incomplete - regenerating")
        cluster_labels_dict = None
else:
    cluster_labels_dict = None

if cluster_labels_dict is None:
    show_status(f"Generating labels for {len(cluster_exemplars)} clusters...")

    cluster_labels_dict = {}
    failed_count = 0

    for cluster_id in tqdm(cluster_exemplars.keys(), desc="Labeling"):
        data = cluster_exemplars[cluster_id]
        result = label_cluster(cluster_id, data['exemplars'], data['is_cross_cutting'])
        cluster_labels_dict[str(cluster_id)] = result

        if not result.get('success', False):
            failed_count += 1

        time.sleep(0.3)

    with open(labels_file, 'w') as f:
        json.dump(cluster_labels_dict, f, indent=2)

    show_complete(f"Labeling complete: {len(cluster_labels_dict) - failed_count} successful, {failed_count} failed")

_label_success = sum(1 for v in cluster_labels_dict.values() if v.get('success', False))
_label_total = max(1, len(cluster_labels_dict))
_label_success_rate = _label_success / _label_total
print(f"Cluster-label success rate: {_label_success_rate:.1%} ({_label_success}/{_label_total})")
if _label_success_rate < MIN_CLUSTER_LABEL_SUCCESS_RATE:
    raise RuntimeError(
        f"Cluster-label success rate ({_label_success_rate:.1%}) below threshold "
        f"({MIN_CLUSTER_LABEL_SUCCESS_RATE:.0%}). Re-run cluster labelling or inspect failures "
        f"before continuing; downstream framing-lens analysis depends on these labels."
    )

# Create label list
cluster_labels_list = [cluster_labels_dict.get(str(i), {}).get('label', f'Cluster {i}')
                       for i in range(N_CONCERN_CLUSTERS)]

# Summary
print("\nTop 15 Clusters by Size:")
cluster_summary = []
for cluster_id, data in cluster_exemplars.items():
    label = cluster_labels_dict.get(str(cluster_id), {}).get('label', f'Cluster {cluster_id}')
    cluster_summary.append({
        'cluster_id': cluster_id,
        'label': label,
        'size': data['size'],
        'entropy': data['entropy'],
        'type': 'Cross-cutting' if data['is_cross_cutting'] else 'Tech-specific'
    })

summary_df = pd.DataFrame(cluster_summary).sort_values('size', ascending=False)
print(summary_df[['cluster_id', 'label', 'size', 'type']].head(15).to_string(index=False))

summary_df.to_csv(OUTPUT_FOLDER / "cluster_summary.csv", index=False)


---
# SECTION 5: Framing lens analysis
---

In [ ]:
# @title Identify framing lenses

show_status("Generating framing lens suggestions...")

# Prepare cluster summary for GPT
cluster_info = []
for cluster_id, data in cluster_exemplars.items():
    label = cluster_labels_dict.get(str(cluster_id), {}).get('label', f'Cluster {cluster_id}')
    description = cluster_labels_dict.get(str(cluster_id), {}).get('description', '')
    cluster_type = 'cross-cutting' if data['is_cross_cutting'] else 'tech-specific'
    cluster_info.append(f"{cluster_id}. {label} ({cluster_type}, n={data['size']}): {description}")

cluster_summary_text = "\n".join(cluster_info)

lens_prompt = f"""Analyze these {N_CONCERN_CLUSTERS} concern clusters from UK public dialogue reports.

Clusters:
{cluster_summary_text}

Group these clusters into 8-12 higher-level FRAMING LENSES that capture how publics frame their concerns.

For each lens provide:
1. Name (2-4 words)
2. Description (1 sentence)
3. List of cluster IDs that belong to this lens

A cluster can belong to multiple lenses if appropriate.
Ensure all clusters are assigned to at least one lens.

Return JSON:
{{"framing_lenses": [
  {{"name": "...", "description": "...", "suggested_clusters": [0, 1, 2...]}},
  ...
]}}"""

try:
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": "Expert in public engagement and discourse analysis. Return only valid JSON."},
            {"role": "user", "content": lens_prompt}
        ],
        max_completion_tokens=8000,
    )

    content = response.choices[0].message.content
    if '```' in content:
        parts = content.split('```')
        for part in parts:
            if part.startswith('json'):
                content = part[4:].strip()
                break
            elif part.strip().startswith('{'):
                content = part.strip()
                break

    suggested_lenses = json.loads(content)

    print("Suggested Framing Lenses:\n")
    for lens in suggested_lenses['framing_lenses']:
        n_clusters = len(lens['suggested_clusters'])
        print(f"  {lens['name']}")
        print(f"    {lens['description']}")
        print(f"    Clusters: {lens['suggested_clusters'][:8]}...\n")

    show_complete(f"Generated {len(suggested_lenses['framing_lenses'])} framing lenses")

except Exception as e:
    print(f"Error generating lenses: {e}")
    suggested_lenses = {'framing_lenses': []}


In [ ]:
# @title Map concern clusters to framing lenses

FRAMING_LENS_MAPPINGS = {}
for lens in suggested_lenses['framing_lenses']:
    FRAMING_LENS_MAPPINGS[lens['name']] = {
        'description': lens['description'],
        'cluster_ids': lens['suggested_clusters']
    }

with open(OUTPUT_FOLDER / "framing_lens_mappings.json", 'w') as f:
    json.dump(FRAMING_LENS_MAPPINGS, f, indent=2)

# Map clusters to lenses
cluster_to_lens = {}
for lens_name, data in FRAMING_LENS_MAPPINGS.items():
    for cluster_id in data['cluster_ids']:
        if cluster_id not in cluster_to_lens:
            cluster_to_lens[cluster_id] = []
        cluster_to_lens[cluster_id].append(lens_name)

all_cluster_ids = set(range(N_CONCERN_CLUSTERS))
covered_cluster_ids = set(cluster_to_lens.keys())
uncovered = all_cluster_ids - covered_cluster_ids
if uncovered:
    print("Uncovered cluster IDs:", sorted(uncovered)[:30])
    raise RuntimeError(
        f"{len(uncovered)} of {N_CONCERN_CLUSTERS} concern clusters are not assigned to any "
        f"framing lens. The lens-suggestion step did not honour the 'all clusters covered' "
        f"requirement. Re-run the framing-lens identification cell or assign the missing "
        f"clusters manually before continuing."
    )

print("Framing Lens Mappings:")
for lens_name, data in FRAMING_LENS_MAPPINGS.items():
    n_clusters = len(data['cluster_ids'])
    # Count concerns in this lens
    lens_mask = concerns_df['cluster_id'].isin(data['cluster_ids'])
    n_concerns = lens_mask.sum()
    print(f"  {lens_name}: {n_clusters} clusters, {n_concerns} concerns")


In [ ]:
# @title Compute framing lens prevalence

show_status("Computing framing lens prevalence...")

lens_prevalence = {}
for lens_name, data in FRAMING_LENS_MAPPINGS.items():
    lens_mask = concerns_df['cluster_id'].isin(data['cluster_ids'])
    lens_prevalence[lens_name] = lens_mask.sum()

lens_prev_df = pd.DataFrame([
    {'Framing Lens': k, 'Concerns': v}
    for k, v in lens_prevalence.items()
]).sort_values('Concerns', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(lens_prev_df['Framing Lens'], lens_prev_df['Concerns'], color='steelblue')
ax.set_xlabel('Number of Concern Phrases')
ax.set_title('Framing Lens Prevalence Across All Documents')

for bar, val in zip(bars, lens_prev_df['Concerns']):
    ax.text(val + 10, bar.get_y() + bar.get_height()/2, f'{val:,}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "framing_lens_prevalence.png", dpi=150, bbox_inches='tight')
plt.show()

show_complete("Framing lens prevalence computed")

In [ ]:
# @title Hierarchical structure of concern clusters (dendrogram)
# Validates the framing-lens scheme by showing whether concern clusters
# that the LLM grouped into the same lens also cluster together in a
# data-driven hierarchy. Strong correspondence supports the lens scheme;
# divergence is a finding worth reporting.

import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist

show_status("Computing concern-cluster dendrogram...")

# L2-normalise centroids (cosine distance via euclidean on normed vectors)
_norm = np.linalg.norm(centroids, axis=1, keepdims=True)
centroids_normed = centroids / np.clip(_norm, 1e-12, None)

# Pairwise cosine distance and average linkage
distances = pdist(centroids_normed, metric="cosine")
Z = linkage(distances, method="average")

# Build leaf labels from existing cluster labels
cluster_label_lookup = {
    int(k): v.get("label", f"Cluster {k}")
    for k, v in cluster_labels_dict.items()
}
leaf_labels = [cluster_label_lookup.get(i, f"Cluster {i}")
               for i in range(len(centroids_normed))]

# Build cluster->lens lookup
cluster_to_lens = {}
for lens_name, info in FRAMING_LENS_MAPPINGS.items():
    for cid in info.get("cluster_ids", []):
        cluster_to_lens[int(cid)] = lens_name

# Assign a colour per lens
import matplotlib.cm as cm
lens_names = list(FRAMING_LENS_MAPPINGS.keys())
lens_colours = {name: cm.tab20(i / max(len(lens_names), 1))
                for i, name in enumerate(lens_names)}

# Plot
fig, ax = plt.subplots(figsize=(11, max(8, len(leaf_labels) * 0.18)))
dd = dendrogram(
    Z,
    labels=leaf_labels,
    orientation="left",
    leaf_font_size=8,
    color_threshold=0,
    above_threshold_color="grey",
    ax=ax,
)
# Recolour leaf labels by lens
ylabels = ax.get_ymajorticklabels()
for label in ylabels:
    cid_label = label.get_text()
    matched_cid = next((cid for cid, name in cluster_label_lookup.items()
                        if name == cid_label), None)
    if matched_cid is not None:
        lens = cluster_to_lens.get(matched_cid, None)
        if lens is not None:
            label.set_color(lens_colours.get(lens, "black"))

ax.set_xlabel("Cosine distance between concern cluster centroids")
ax.set_title("Hierarchical structure of concern clusters\n(leaf colour = LLM-assigned framing lens)")

from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=col, label=name) for name, col in lens_colours.items()]
ax.legend(handles=legend_handles, loc="lower right", fontsize=7, title="Framing lens")

plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "concern_cluster_dendrogram.png", dpi=200, bbox_inches="tight")
plt.show()

# Quantitative comparison: ARI between LLM lens assignment and dendrogram cut
from sklearn.metrics import adjusted_rand_score

n_lenses = len(FRAMING_LENS_MAPPINGS)
data_driven_groups = fcluster(Z, t=n_lenses, criterion="maxclust")
lens_assignment = np.array([
    lens_names.index(cluster_to_lens.get(i, lens_names[0]))
    if cluster_to_lens.get(i) in lens_names else -1
    for i in range(len(centroids_normed))
])
mask = lens_assignment >= 0
ari = adjusted_rand_score(lens_assignment[mask], data_driven_groups[mask])
print(f"\nNumber of concern lenses (LLM-derived): {n_lenses}")
print(f"Adjusted Rand Index between lens assignment and dendrogram cut at "
      f"{n_lenses} groups: {ari:.3f}")
print("Higher = stronger correspondence between data-driven hierarchy and "
      "LLM-derived lens scheme.")

show_complete("Concern-cluster dendrogram complete.")

# SECTION 6: Shared concerns, shared framings, and AI distinctiveness
This section has three goals:

**6.1 Shared structure (all technologies, document‑weighted):** What concerns and framings recur across technologies?

**6.2 AI vs non‑AI baseline (technology‑weighted):** How does AI's profile differ from the average *non‑AI technology* (to avoid circularity, because AI documents are a large share of the corpus)?

**6.3 Distinctiveness:** Which concerns / framings are over‑ or under‑represented in AI relative to the non‑AI baseline?

**Important data contract:** all technology labels come exclusively from the metadata file (see `technology_meta`). LLM‑generated cluster descriptions must never be used as technology categories.


In [ ]:
# --- Section 6 prerequisites (canonical tech + cross-cutting clusters) ---

import numpy as np

TECH_COL = "technology_meta"

# v16-fix: compute normalised entropy locally rather than relying on a global
# called `normalized_entropy` (which is a function in this notebook, not a dict).
# This mirrors the same fix in the stable-core figure cell.
n_techs_for_norm = concerns_df[TECH_COL].nunique()
max_ent_for_norm = np.log(n_techs_for_norm) if n_techs_for_norm > 1 else 1.0
_norm_ent_for_xcut = {cid: (e / max_ent_for_norm) for cid, e in cluster_entropy.items()}

CROSS_CUTTING_ENTROPY_THRESHOLD = 0.5
cross_cutting_cluster_ids = {
    cid for cid, ent in _norm_ent_for_xcut.items()
    if ent >= CROSS_CUTTING_ENTROPY_THRESHOLD
}

print(f"Cross-cutting clusters: {len(cross_cutting_cluster_ids)} of {len(cluster_entropy)} "
      f"(normalised entropy >= {CROSS_CUTTING_ENTROPY_THRESHOLD})")

In [ ]:
# @title Shared concerns across technologies (document-weighted)

show_status("Computing shared concern structure (all technologies)...")

TECH_COL = "technology_meta"

# Canonical list of technologies from metadata
technologies = sorted(concerns_df[TECH_COL].dropna().unique().tolist())

# Salience matrix: rows=technology, cols=cluster_id, values=proportion of concerns
salience_rows = {}
for tech in technologies:
    tech_mask = concerns_df[TECH_COL] == tech
    tech_total = int(tech_mask.sum())
    if tech_total == 0:
        continue
    counts = concerns_df.loc[tech_mask, "cluster_id"].value_counts()
    salience_rows[tech] = (counts / tech_total).to_dict()

salience_df = pd.DataFrame(salience_rows).T.fillna(0)

# Document-weighted global prevalence of clusters
global_cluster_prevalence = concerns_df["cluster_id"].value_counts(normalize=True)

# Cross-cutting metric (entropy over technologies) computed earlier as cluster_entropy
cluster_meta_df = (
    pd.DataFrame(
        {
            "cluster_id": list(cluster_entropy.keys()),
            "tech_entropy": [cluster_entropy[c] for c in cluster_entropy.keys()],
            "global_prevalence": [global_cluster_prevalence.get(c, 0) for c in cluster_entropy.keys()],
        }
    )
    .set_index("cluster_id")
    .sort_values(["global_prevalence", "tech_entropy"], ascending=False)
)

# Shared concerns = high prevalence + high entropy
shared_concerns = cluster_meta_df.head(20)

print("Top shared concern clusters (high prevalence + cross-technology spread):")
display(shared_concerns.head(12))

# Quick bar chart (prevalence)
plt.figure(figsize=(10, 5))
shared_concerns.sort_values("global_prevalence")["global_prevalence"].plot(kind="barh", legend=False)
plt.title("Shared concerns across technologies (top 20 clusters by prevalence × spread)")
plt.xlabel("Share of all extracted concern phrases")
plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "shared_concerns_top20.png")
plt.show()

shared_concerns.reset_index().to_csv(OUTPUT_FOLDER / "shared_concerns_top20.csv", index=False)

show_complete("Shared concerns computed")


In [ ]:
# @title Shared framings across technologies (document-weighted)

show_status("Computing shared framing structure (all technologies)...")

TECH_COL = 'technology_meta'
technologies = sorted(concerns_df[TECH_COL].dropna().unique().tolist())

# Lens prevalence overall + entropy across technologies
lens_stats = []
for lens_name, data in FRAMING_LENS_MAPPINGS.items():
    lens_clusters = set(data['cluster_ids'])
    lens_mask = concerns_df['cluster_id'].isin(lens_clusters)
    overall_prev = lens_mask.mean()  # document-weighted (each concern phrase counts equally)

    # Technology distribution within lens
    tech_counts = concerns_df.loc[lens_mask, TECH_COL].value_counts()
    tech_probs = (tech_counts / tech_counts.sum()).values if tech_counts.sum() > 0 else np.array([])
    lens_entropy = float(entropy(tech_probs)) if len(tech_probs) > 1 else 0.0

    lens_stats.append({
        'framing_lens': lens_name,
        'overall_prevalence': float(overall_prev),
        'tech_entropy': lens_entropy,
        'n_concerns_in_lens': int(lens_mask.sum())
    })

lens_meta_df = (pd.DataFrame(lens_stats)
                .sort_values(['overall_prevalence','tech_entropy'], ascending=False))

print("Shared framings (high prevalence + cross-technology spread):")
display(lens_meta_df.head(12))

# Scatter: prevalence vs entropy
plt.figure(figsize=(7,5))
plt.scatter(lens_meta_df['tech_entropy'], lens_meta_df['overall_prevalence'])
for _, r in lens_meta_df.iterrows():
    plt.text(r['tech_entropy'], r['overall_prevalence'], r['framing_lens'], fontsize=8)
plt.xlabel("Entropy across technologies")
plt.ylabel("Share of all extracted concerns")
plt.title("Shared framings across technologies")
plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "shared_framings_scatter.png")
plt.show()

lens_meta_df.to_csv(OUTPUT_FOLDER / "shared_framings.csv", index=False)
show_complete("Shared framings computed")


In [ ]:
# @title Compare AI and non-AI dialogues by framing lens

show_status("Computing AI vs non-AI framing profile (technology-weighted baseline)...")

TECH_COL = 'technology_meta'
technologies = sorted(concerns_df[TECH_COL].dropna().unique().tolist())
non_ai_techs = [t for t in technologies if t != 'AI']

# Lens salience per technology (within-technology proportions)
lens_salience_by_tech = {}
for tech in technologies:
    tech_mask = concerns_df[TECH_COL] == tech
    tech_total = tech_mask.sum()
    if tech_total == 0:
        continue
    lens_salience_by_tech[tech] = {}
    for lens_name, data in FRAMING_LENS_MAPPINGS.items():
        lens_mask = concerns_df['cluster_id'].isin(data['cluster_ids'])
        lens_salience_by_tech[tech][lens_name] = float((tech_mask & lens_mask).sum() / tech_total)

lens_salience_df = pd.DataFrame(lens_salience_by_tech).T.fillna(0)

# Technology-weighted non-AI baseline (each non-AI technology counts equally)
non_ai_avg = lens_salience_df.loc[non_ai_techs].mean(axis=0) if len(non_ai_techs) else pd.Series(dtype=float)

# Plot radar: AI vs non-AI avg
if 'AI' in lens_salience_df.index and len(non_ai_avg) > 0:
    categories = list(non_ai_avg.index)
    ai_vals = lens_salience_df.loc['AI', categories].tolist()
    base_vals = non_ai_avg[categories].tolist()

    # Close the loop
    categories_loop = categories + [categories[0]]
    ai_loop = ai_vals + [ai_vals[0]]
    base_loop = base_vals + [base_vals[0]]

    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(r=ai_loop, theta=categories_loop, fill='toself', name='AI'))
    fig.add_trace(go.Scatterpolar(r=base_loop, theta=categories_loop, fill='toself', name='Non‑AI average (tech‑weighted)'))
    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, max(ai_loop+base_loop)*1.1])),
        title="AI framing profile vs non‑AI baseline (technology‑weighted)"
    )
    fig.write_html(OUTPUT_FOLDER / "ai_vs_nonai_framing_radar.html")
# v18: also save a static PNG copy for paper use (requires kaleido package)
try:
    fig.write_image(OUTPUT_FOLDER / "ai_vs_nonai_framing_radar.png", width=900, height=700, scale=2)
except Exception as _e:
    print(f"Note: PNG export failed ({_e}); HTML version still saved.")
    print("If you need PNG, install kaleido: !pip install -q kaleido")
    fig.show()

# Save table
lens_salience_df.to_csv(OUTPUT_FOLDER / "lens_salience_by_technology_meta.csv")
non_ai_avg.to_frame('non_ai_avg_tech_weighted').to_csv(OUTPUT_FOLDER / "non_ai_avg_framing_profile.csv")
show_complete("AI vs non-AI framing profile computed")


In [ ]:
# @title (v16) Document-weighted AI vs non-AI baseline (concerns) — diagnostic
# Issue 10 partial fix from the audit: the canonical AI vs non-AI comparison
# uses a TECHNOLOGY-WEIGHTED baseline (each non-AI technology contributes
# equally to the baseline), which gives small-N technologies like Quantum
# (6 phrases) the same weight as Genetic technology (675 phrases). This is
# the right move conceptually for "what does the average non-AI dialogue
# look like", but it can produce noisy baselines.
#
# This cell computes a DOCUMENT-WEIGHTED baseline as a sanity check: the
# baseline is simply "what proportion of all non-AI concern phrases fall in
# each lens?". This is dominated by Genetic technology, which is the
# largest non-AI corpus by volume. Compare this side-by-side with the
# technology-weighted baseline above; if conclusions about AI distinctiveness
# only hold under one weighting, that is a finding worth reporting.

show_status("Computing document-weighted non-AI baseline...")

ai_mask = concerns_df["technology_meta"] == "AI"
non_ai_mask = ~ai_mask

# Document-weighted: pool all non-AI phrases, count lens shares directly
non_ai_lens_counts = {}
for lens_name, data in FRAMING_LENS_MAPPINGS.items():
    lens_mask = concerns_df["cluster_id"].isin(data["cluster_ids"])
    non_ai_lens_counts[lens_name] = int((non_ai_mask & lens_mask).sum())

non_ai_total = int(non_ai_mask.sum())
non_ai_doc_weighted = {k: v / non_ai_total for k, v in non_ai_lens_counts.items()} if non_ai_total else {}

# AI shares (unchanged)
ai_lens_shares = {}
ai_total = int(ai_mask.sum())
for lens_name, data in FRAMING_LENS_MAPPINGS.items():
    lens_mask = concerns_df["cluster_id"].isin(data["cluster_ids"])
    ai_lens_shares[lens_name] = float((ai_mask & lens_mask).sum() / ai_total) if ai_total else 0.0

# Combine
doc_weighted_comparison = pd.DataFrame({
    "lens": list(FRAMING_LENS_MAPPINGS.keys()),
    "ai_share": [ai_lens_shares[l] for l in FRAMING_LENS_MAPPINGS],
    "non_ai_doc_weighted_share": [non_ai_doc_weighted.get(l, 0.0) for l in FRAMING_LENS_MAPPINGS],
})
doc_weighted_comparison["ai_minus_doc_weighted"] = (
    doc_weighted_comparison["ai_share"] - doc_weighted_comparison["non_ai_doc_weighted_share"]
)
doc_weighted_comparison = doc_weighted_comparison.sort_values("ai_minus_doc_weighted", ascending=False)

doc_weighted_comparison.to_csv(OUTPUT_FOLDER / "ai_vs_nonai_lens_doc_weighted.csv", index=False)

print("AI vs non-AI by lens (document-weighted baseline):")
print(doc_weighted_comparison.to_string(index=False))
print()
print("Compare with the technology-weighted version above. If the rank order")
print("of AI-distinctive lenses changes substantially between the two, the")
print("AI distinctiveness claim is sensitive to baseline construction.")

show_complete("Document-weighted baseline complete")


In [ ]:
# @title Compare AI and non-AI dialogues on cross-cutting concerns

show_status("Computing AI vs non-AI concern profile on cross-cutting clusters...")

TECH_COL = 'technology_meta'
technologies = sorted(concerns_df[TECH_COL].dropna().unique().tolist())
non_ai_techs = [t for t in technologies if t != 'AI']

# Use cross_cutting_labels from SECTION 3 (derived from entropy threshold)
cross_cutting_cluster_ids = set(cross_cutting_labels.keys())

# Technology profiles over cross-cutting clusters
profiles = {}
for tech in technologies:
    tech_mask = concerns_df[TECH_COL] == tech
    tech_total = tech_mask.sum()
    if tech_total == 0:
        continue
    counts = concerns_df.loc[tech_mask & concerns_df['cluster_id'].isin(cross_cutting_cluster_ids), 'cluster_id'].value_counts()
    # normalise over all concerns for that tech (so "attention share" to cross-cutting clusters)
    profiles[tech] = (counts / tech_total).to_dict()

profiles_df = pd.DataFrame(profiles).T.fillna(0)

non_ai_avg = profiles_df.loc[non_ai_techs].mean(axis=0) if len(non_ai_techs) else pd.Series(dtype=float)

# Save for later use
profiles_df.to_csv(OUTPUT_FOLDER / "cross_cutting_cluster_profiles_by_tech.csv")
non_ai_avg.to_frame('non_ai_avg_tech_weighted').to_csv(OUTPUT_FOLDER / "cross_cutting_cluster_profile_non_ai_avg.csv")

show_complete("Cross-cutting concern profiles computed")


In [ ]:
# --- Build CLUSTER_LABELS dict for plots (cluster_id -> label) ---
if "summary_df" in globals() and {"cluster_id", "label"}.issubset(summary_df.columns):
    CLUSTER_LABELS = dict(zip(summary_df["cluster_id"], summary_df["label"]))
else:
    CLUSTER_LABELS = {}


In [ ]:
# @title Visualise AI concern fingerprint (AI vs non-AI)

TECH_COL = 'technology_meta'

if 'AI' in profiles_df.index and len(non_ai_avg) > 0:
    show_status("Creating AI fingerprint radar (AI vs non-AI)...")

    diff = (profiles_df.loc['AI'] - non_ai_avg).sort_values()
    top_low = diff.head(5).index.tolist()
    top_high = diff.tail(7).index.tolist()
    selected = top_low + top_high

    # Human-friendly labels
    def pretty_cluster(cid):
        return CLUSTER_LABELS.get(cid, f"Cluster {cid}")

    categories = [pretty_cluster(c) for c in selected]
    ai_vals = profiles_df.loc['AI', selected].tolist()
    base_vals = non_ai_avg[selected].tolist()

    categories_loop = categories + [categories[0]]
    ai_loop = ai_vals + [ai_vals[0]]
    base_loop = base_vals + [base_vals[0]]

    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(r=ai_loop, theta=categories_loop, fill='toself', name='AI'))
    fig.add_trace(go.Scatterpolar(r=base_loop, theta=categories_loop, fill='toself', name='Non‑AI average (tech‑weighted)'))
    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, max(ai_loop+base_loop)*1.1])),
        title="AI concern fingerprint vs non‑AI baseline (cross‑cutting clusters)"
    )
    fig.write_html(OUTPUT_FOLDER / "ai_vs_nonai_concern_radar.html")
# v18: also save a static PNG copy for paper use (requires kaleido package)
try:
    fig.write_image(OUTPUT_FOLDER / "ai_vs_nonai_concern_radar.png", width=900, height=700, scale=2)
except Exception as _e:
    print(f"Note: PNG export failed ({_e}); HTML version still saved.")
    print("If you need PNG, install kaleido: !pip install -q kaleido")
    fig.show()

show_complete("AI fingerprint radar created")


In [ ]:
# @title Identify distinctive AI concerns and framings

show_status("Computing distinctive concerns and framings for AI...")

TECH_COL = 'technology_meta'
non_ai_techs = [t for t in technologies if t != 'AI']

# Distinctive concern clusters (all clusters, tech-weighted baseline)
all_cluster_profiles = salience_df  # from 6.1a (rows=tech, cols=cluster_id)

if 'AI' in all_cluster_profiles.index and len(non_ai_techs) > 0:
    non_ai_avg_all = all_cluster_profiles.loc[non_ai_techs].mean(axis=0)
    ai_vs_base = (all_cluster_profiles.loc['AI'] - non_ai_avg_all).sort_values()

    most_under = ai_vs_base.head(10)
    most_over = ai_vs_base.tail(10)

    def label(cid):
        return CLUSTER_LABELS.get(cid, f"Cluster {cid}")

    distinctive_concerns = pd.DataFrame({
        'cluster_id': list(most_under.index) + list(most_over.index),
        'cluster_label': [label(c) for c in list(most_under.index) + list(most_over.index)],
        'ai_minus_nonai': list(most_under.values) + list(most_over.values),
        'ai_share': [float(all_cluster_profiles.loc['AI', c]) for c in list(most_under.index) + list(most_over.index)],
        'nonai_share': [float(non_ai_avg_all[c]) for c in list(most_under.index) + list(most_over.index)],
    }).sort_values('ai_minus_nonai')

    display(distinctive_concerns)
    distinctive_concerns.to_csv(OUTPUT_FOLDER / "ai_distinctive_concerns.csv", index=False)

# Distinctive framings (lens level)
if 'AI' in lens_salience_df.index and len(non_ai_techs) > 0:
    non_ai_avg_lens = lens_salience_df.loc[non_ai_techs].mean(axis=0)
    lens_diff = (lens_salience_df.loc['AI'] - non_ai_avg_lens).sort_values()

    df = pd.DataFrame({
        'framing_lens': lens_diff.index,
        'ai_minus_nonai': lens_diff.values,
        'ai_share': lens_salience_df.loc['AI', lens_diff.index].values,
        'nonai_share': non_ai_avg_lens[lens_diff.index].values
    }).sort_values('ai_minus_nonai')

    display(df.head(10))
    display(df.tail(10))
    df.to_csv(OUTPUT_FOLDER / "ai_distinctive_framings.csv", index=False)

show_complete("Distinctiveness outputs saved")


In [ ]:
# pretty_label — imported from dialogue_utils (see Cell 5)
pass

---
# SECTION 7: AI concern dynamics over time
---

In [ ]:
# normalized_entropy, hhi, topk_share, parse_year — imported from dialogue_utils (see Cell 5)
pass

In [ ]:
# @title (v16) Wider-time-window AI concern analysis
# Issue 4 partial fix from the audit: year-by-year AI concern data is
# extremely sparse for some years (e.g. 2024 had 9 phrases in v15). With
# small N, normalised entropy is mathematically forced toward its maximum
# regardless of the underlying distribution, confounding any temporal-trend
# claim about "concern diversity" with sample size.
#
# This cell repeats the temporal AI analysis using 4 wider windows
# (2004-2017, 2018-2020, 2021-2023, 2024-2025) so each window contains
# enough data to make the entropy/share numbers interpretable. The
# year-by-year analysis is retained above; treat this as the primary table
# for any quantitative temporal claim in the paper.

show_status("Running wider-window temporal AI analysis...")

def assign_window(year):
    if pd.isna(year):
        return None
    y = int(year)
    if y <= 2017:
        return "2004-2017"
    if y <= 2020:
        return "2018-2020"
    if y <= 2023:
        return "2021-2023"
    return "2024-2025"

ai_concerns_window = concerns_df[concerns_df["technology_meta"] == "AI"].copy()
ai_concerns_window["time_window"] = ai_concerns_window["year"].apply(assign_window)
ai_concerns_window = ai_concerns_window.dropna(subset=["time_window"])

# Per-window: total phrases, cluster distribution, entropy
window_summary = []
for window, group in ai_concerns_window.groupby("time_window"):
    n_phrases = len(group)
    cluster_counts = group["cluster_id"].value_counts()
    probs = (cluster_counts / cluster_counts.sum()).values if cluster_counts.sum() > 0 else np.array([])
    raw_ent = float(entropy(probs)) if len(probs) > 1 else 0.0
    norm_ent = raw_ent / np.log(len(cluster_counts)) if len(cluster_counts) > 1 else 0.0
    window_summary.append({
        "time_window": window,
        "n_phrases": n_phrases,
        "n_clusters_present": int((cluster_counts > 0).sum()),
        "n_documents": int(group["source_file"].nunique()),
        "raw_entropy": raw_ent,
        "normalized_entropy_within_window": norm_ent,
    })

window_summary_df = pd.DataFrame(window_summary).sort_values("time_window")
window_summary_df.to_csv(OUTPUT_FOLDER / "ai_concern_window_summary.csv", index=False)

print("Wider-window AI concern summary:")
print(window_summary_df.to_string(index=False))
print()
print("Note: 'normalized_entropy_within_window' is normalised by the number of")
print("clusters that contain any phrases in that window — so it is bounded by")
print("the data, not by the absolute cluster count. With more data per window,")
print("these values are more interpretable than the year-by-year ones above.")

# Also: per-window top clusters (qualitative)
top_clusters_per_window = []
for window, group in ai_concerns_window.groupby("time_window"):
    counts = group["cluster_id"].value_counts().head(10)
    for cid, n in counts.items():
        label = CLUSTER_LABELS.get(cid, f"Cluster {cid}") if "CLUSTER_LABELS" in globals() else f"Cluster {cid}"
        top_clusters_per_window.append({
            "time_window": window,
            "cluster_id": cid,
            "label": label,
            "n_phrases": int(n),
            "share_within_window": float(n) / len(group),
        })

top_window_df = pd.DataFrame(top_clusters_per_window)
top_window_df.to_csv(OUTPUT_FOLDER / "ai_concern_window_top_clusters.csv", index=False)
print(f"\nWrote top-10-clusters-per-window table to ai_concern_window_top_clusters.csv")

show_complete("Wider-window analysis complete")


In [ ]:
# @title Visualise AI concern trajectory over time

import matplotlib.pyplot as plt
import numpy as np

# Use traj DataFrame from the previous cell
# columns: year, pc1, pc2

traj2 = traj.sort_values("year").reset_index(drop=True)

# Center trajectory on first year
x0, y0 = traj2.loc[0, ["pc1", "pc2"]]
dx = traj2["pc1"] - x0
dy = traj2["pc2"] - y0

years = traj2["year"].to_numpy()

plt.figure(figsize=(6, 6))
sc = plt.scatter(dx, dy, c=years, cmap="viridis", s=80)
plt.plot(dx, dy, alpha=0.6)

plt.axhline(0, color="grey", linewidth=1, alpha=0.5)
plt.axvline(0, color="grey", linewidth=1, alpha=0.5)

plt.xlabel("Displacement in concern space (PC1)")
plt.ylabel("Displacement in concern space (PC2)")
plt.title("AI concern profile over time\n(displacement from initial position)")

cbar = plt.colorbar(sc)
cbar.set_label("Year")

plt.tight_layout()
plt.show()


In [ ]:
# @title Analyse AI concern salience trajectories
# Shows which concern clusters rise or fall most within AI discourse over time.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ai_year and years already exist from previous cells
# ai_year: years × clusters (raw salience)

# Normalize within each year so values sum to 1 (relative attention)
ai_rel = ai_year.div(ai_year.sum(axis=1), axis=0).fillna(0)

# Compute linear trend (slope) for each cluster
# Using simple OLS slope over time
t = np.arange(len(ai_rel))  # time index (no assumption of equal year gaps)
slopes = {}
for c in ai_rel.columns:
    y = ai_rel[c].to_numpy()
    if np.all(y == 0):
        slopes[c] = np.nan
    else:
        # slope of y ~ t
        slopes[c] = np.polyfit(t, y, 1)[0]

trend = pd.Series(slopes).dropna().sort_values(ascending=False)

# Select top rising and declining clusters
N = 8
top_up = trend.head(N)
top_down = trend.tail(N)

# Plot trajectories
plt.figure(figsize=(11, 5))

# Rising
plt.subplot(1, 2, 1)
for c in top_up.index:
    plt.plot(ai_rel.index, ai_rel[c], marker="o", label=c)
plt.title("AI concerns increasing in relative salience")
plt.xlabel("Year")
plt.ylabel("Share of AI attention")
plt.legend(fontsize=9)

# Declining
plt.subplot(1, 2, 2)
for c in top_down.index:
    plt.plot(ai_rel.index, ai_rel[c], marker="o", label=c)
plt.title("AI concerns decreasing in relative salience")
plt.xlabel("Year")
plt.ylabel("Share of AI attention")
plt.legend(fontsize=9)

plt.tight_layout()
plt.show()

display(
    pd.DataFrame({
        "slope": trend
    }).assign(direction=lambda d: np.where(d["slope"] > 0, "rising", "declining"))
)


In [ ]:
# @title Visualise AI concern salience over time
# Rows = concern clusters
# Columns = years
# Values = relative salience within AI discourse (row-normalised per year)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ai_year already exists: years × clusters (raw salience)
# years is a sorted array of years

# 1) Normalise within each year (relative attention)
ai_rel = ai_year.div(ai_year.sum(axis=1), axis=0).fillna(0)

# 2) Select clusters to display
# Option A: top N by overall AI salience
N = 20
top_clusters = ai_rel.sum(axis=0).sort_values(ascending=False).head(N).index

heat = ai_rel[top_clusters]

# 3) Order clusters by when they peak (helps visual interpretation)
peak_year_idx = heat.values.argmax(axis=0)
order = np.argsort(peak_year_idx)
heat = heat.iloc[:, order]

# Transpose so clusters are rows
heat_T = heat.T

# 4) Plot heat map
plt.figure(figsize=(12, 7))
im = plt.imshow(
    heat_T,
    aspect="auto",
    cmap="viridis"
)

plt.colorbar(im, label="Share of AI attention (within year)")

plt.yticks(
    ticks=np.arange(len(heat_T.index)),
    labels=heat_T.index
)
plt.xticks(
    ticks=np.arange(len(heat_T.columns)),
    labels=heat_T.columns,
    rotation=45
)

plt.xlabel("Year")
plt.ylabel("Concern cluster")
plt.title("AI public concerns over time\n(relative salience within AI discourse)")

plt.tight_layout()
plt.show()


---
# SECTION 8: The stable core of public technology concerns
---

In [ ]:
# @title Visualise the stable core of public concerns

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# ---- prerequisites ----
if "cluster_entropy" not in globals():
    raise NameError("cluster_entropy not found. Run the cluster entropy section first.")
if "concerns_df" not in globals():
    raise NameError("concerns_df not found. Run concern extraction first.")

TECH_COL = "technology_meta"
ENTROPY_THRESHOLD = 0.5  # canonical cross-cutting threshold (applied to NORMALISED entropy)

# ---- build cluster-level dataframe ----
# v16-fix: my earlier patch assumed `normalized_entropy` was a dict, but it's
# actually a function defined elsewhere in the notebook. Compute normalised
# entropy locally from the raw `cluster_entropy` dict instead.
n_techs = concerns_df[TECH_COL].nunique()
max_entropy = np.log(n_techs) if n_techs > 1 else 1.0
norm_ent_dict = {cid: (e / max_entropy) for cid, e in cluster_entropy.items()}

# cluster size = number of concern phrases
cluster_size = concerns_df["cluster_id"].value_counts()

df = (
    pd.DataFrame({
        "cluster_id": list(norm_ent_dict.keys()),
        "entropy": [norm_ent_dict[c] for c in norm_ent_dict.keys()],
    })
    .set_index("cluster_id")
)
df["size"] = cluster_size
df["size"] = df["size"].fillna(0)
df = df[df["size"] > 0]

# cluster labels (if available)
if "CLUSTER_LABELS" in globals():
    df["label"] = df.index.map(lambda c: CLUSTER_LABELS.get(c, f"Cluster {c}"))
else:
    df["label"] = df.index.map(lambda c: f"Cluster {c}")

entropy_vals = df["entropy"].to_numpy(dtype=float)
size = df["size"].to_numpy(dtype=float)
labels = df["label"].to_numpy(dtype=str)

# ---- define cross-cutting + stable core ----
cross_cutting = entropy_vals >= ENTROPY_THRESHOLD
size_thresh = np.quantile(size, 0.75)  # top 25% by frequency
stable_core = cross_cutting & (size >= size_thresh)

# clusters to annotate (most "core")
score = entropy_vals * np.log1p(size)
annotate_idx = np.argsort(score)[::-1][:12]

# ---- plot ----
plt.figure(figsize=(10, 7))
ax = plt.gca()

# shaded stable-core region
core_region = Rectangle(
    (ENTROPY_THRESHOLD, size_thresh),
    1.0 - ENTROPY_THRESHOLD,
    size.max() * 1.05 - size_thresh,
    alpha=0.12,
    zorder=0
)
ax.add_patch(core_region)
ax.text(
    ENTROPY_THRESHOLD + 0.01,
    size.max() * 1.02,
    "Stable core\n(cross-cutting + high frequency)",
    fontsize=10,
    va="top"
)

plt.scatter(entropy_vals[~cross_cutting], size[~cross_cutting], alpha=0.6, s=35, label="More tech-specific clusters")
plt.scatter(entropy_vals[cross_cutting], size[cross_cutting], alpha=0.8, s=45, label="More cross-cutting clusters")
plt.scatter(entropy_vals[stable_core], size[stable_core], s=90, label="Stable core")

for i in annotate_idx:
    plt.annotate(labels[i], (entropy_vals[i], size[i]), textcoords="offset points", xytext=(6, 4), ha="left", fontsize=9)

plt.axvline(ENTROPY_THRESHOLD, linestyle="--", linewidth=1)
plt.axhline(size_thresh, linestyle="--", linewidth=1)
plt.xlabel("Technology-distribution entropy (higher = more cross-cutting)")
plt.ylabel("Cluster frequency / size (higher = more recurrent)")
plt.title("Figure: The Stable Core of Public Technology Concerns")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "concern_stable_core_scatter.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
# @title Visualise the concern embedding space
# Clusters projected into 2D using PCA over technology-salience vectors.
# Point size ~ cluster frequency proxy; color ~ framing lens (if available).

import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

ARTIFACT_DIR = globals().get("ARTIFACT_DIR", "analysis_artifacts")
EXCEL_PATH = globals().get("EXCEL_PATH", os.path.join(ARTIFACT_DIR, "public_dialogue_analysis_v9.xlsx"))

# 1) Load tech x cluster salience (in-memory)
if "salience_df" not in globals():
    raise NameError("salience_df not found. Run Section 6.1a to compute salience_df first.")

tech_by_cluster = salience_df.copy()
tech_by_cluster = tech_by_cluster.apply(pd.to_numeric, errors="coerce").fillna(0)

# Convert to clusters x technologies
cluster_features = tech_by_cluster.T  # rows=clusters, cols=technologies
clusters = cluster_features.index.astype(str).to_numpy()
X = cluster_features.to_numpy(dtype=float)

print("Cluster feature matrix:", X.shape, "(clusters x technologies)")

# 2) Size proxy (use total salience mass across technologies)
size_vals = cluster_features.sum(axis=1).to_numpy()
size_scaled = 30 + 200 * (np.sqrt(size_vals) / (np.sqrt(size_vals).max() + 1e-9))

# 3) Load framing lens mapping if available (optional)
lens_map = None
json_path = os.path.join(ARTIFACT_DIR, "framing_lens_mappings.json")
if os.path.exists(json_path):
    with open(json_path, "r") as f:
        lens_map = json.load(f)
    # Normalize to {cluster_label: lens}
    if lens_map and all(isinstance(v, list) for v in lens_map.values()):
        inv = {}
        for lens, clist in lens_map.items():
            for c in clist:
                inv[str(c)] = str(lens)
        lens_map = inv
    else:
        lens_map = {str(k): str(v) for k, v in lens_map.items()}

if lens_map is None:
    lens_labels = np.array(["(no lens)"] * len(clusters), dtype=object)
else:
    lens_labels = np.array([lens_map.get(c, "(unmapped)") for c in clusters], dtype=object)

# Encode lenses to integer codes
unique_lenses = pd.unique(lens_labels)
lens_to_code = {lens: i for i, lens in enumerate(unique_lenses)}
codes = np.array([lens_to_code[l] for l in lens_labels], dtype=int)

# 4) PCA projection (standardize across technologies)
X_mean = X.mean(axis=0, keepdims=True)
X_std = X.std(axis=0, keepdims=True) + 1e-9
Xz = (X - X_mean) / X_std

from sklearn.decomposition import PCA
pca = PCA(n_components=2, random_state=0)
coords = pca.fit_transform(Xz)
x, y = coords[:, 0], coords[:, 1]
expl = pca.explained_variance_ratio_

# 5) Plot
plt.figure(figsize=(10, 7))
ax = plt.gca()

# Use a categorical colormap; map integer codes -> colors
cmap = plt.get_cmap("tab20", len(unique_lenses))  # tab20 gives distinct-ish categories
sc = ax.scatter(x, y, s=size_scaled, c=codes, cmap=cmap, alpha=0.85)

ax.set_xlabel(f"Concern space (PC1; {expl[0]:.0%} var)")
ax.set_ylabel(f"Concern space (PC2; {expl[1]:.0%} var)")
ax.set_title("Figure: Concern Space of Public Technology Concerns\n(Clusters projected by technology-salience profile; size ~ frequency proxy)")

# Annotate top-N largest clusters (by size proxy)
topN = 12
top_idx = np.argsort(size_vals)[::-1][:topN]
for i in top_idx:
    ax.annotate(clusters[i], (x[i], y[i]), textcoords="offset points", xytext=(6, 4), ha="left", fontsize=9)

# Legend: show up to 12 most frequent lenses (or all if <= 12)
lens_counts = pd.Series(lens_labels).value_counts()
top_lenses = lens_counts.index[:12].tolist() if len(lens_counts) > 12 else lens_counts.index.tolist()

legend_handles = [
    Line2D([0], [0], marker='o', linestyle='', markersize=8,
           markerfacecolor=cmap(lens_to_code[lens]), markeredgecolor='none', label=lens)
    for lens in top_lenses
]
ax.legend(handles=legend_handles, title="Framing lens (top shown)", loc="best", frameon=True)

plt.tight_layout()
out_path = "figure_concern_space_pca.png"
plt.savefig(out_path, dpi=200)
plt.show()

print("Saved figure to:", out_path)


# PART II: Benefit space (parallel pipeline)

This part repeats the same analytical pipeline as PART I, applied to decontextualised **benefit** phrases.

---


---
# SECTION 2B: Benefit extraction
---

This section extracts decontextualised public benefit phrases from paragraph-level dialogue text. These benefit phrases form the primary analytic units for all subsequent embedding, clustering, and comparative analysis.


In [ ]:
# @title Extract decontextualised benefit phrases

from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
import json
import pandas as pd

show_status("Extracting decontextualised benefit phrases...")

BENEFIT_EXTRACTION_PROMPT = """Extract the core public BENEFITS (upsides, hoped-for gains, opportunities) from this paragraph from a UK public dialogue report.

CRITICAL RULES:
1. Remove ALL technology-specific references (AI, nuclear, genetic, nano, etc.).
2. Extract the underlying benefit that could apply to ANY emerging technology.
3. Keep each benefit phrase concise (3–10 words).
4. Prefer concrete impacts over vague praise (e.g., "faster diagnosis" not "innovation").
5. Do NOT include concerns, caveats, or neutral facts unless they clearly express a benefit.

EXAMPLES:
- "AI could help doctors spot cancers earlier" → "earlier disease detection"
- "Nuclear could provide reliable low-carbon energy" → "reliable low-carbon energy supply"
- "Genetic research might enable personalised treatments" → "more personalised medical treatment"
- "Robots could take on dangerous tasks" → "reduced human exposure to danger"
- "Better public services through improved efficiency" → "more efficient public services"

Return 1–3 benefit phrases, one per line. No bullets, no numbering.
If the paragraph contains no clear public benefit, return "NO_BENEFIT".

Paragraph:
{text}"""

_benefit_extraction_failures = []
_benefit_extraction_failures_lock = threading.Lock()

def _record_benefit_failure(chunk_id, reason):
    with _benefit_extraction_failures_lock:
        _benefit_extraction_failures.append((chunk_id, reason))

_BENEFIT_TECH_TERM_PATTERN = re.compile(
    r"\b("
    r"ai|artificial intelligence|"
    r"nuclear|"
    r"genetic|genome|gene|crispr|embryo|stem cell|"
    r"nano|"
    r"robot|drone|autonomous|"
    r"quantum|"
    r"gm|"
    r"algorithm|algorithmic|automated|automation|"
    r"radiation|engineered|biometric|surveillance|"
    r"neural|brain-computer"
    r")\b",
    re.IGNORECASE,
)

def _benefit_phrase_contains_tech_term(phrase: str) -> bool:
    return bool(_BENEFIT_TECH_TERM_PATTERN.search(phrase or ""))

def extract_benefits_from_paragraph(row_tuple):
    """Extract decontextualised benefits from a single paragraph.

    temperature=1). Logs failures.
    errors, word-boundary regex for the post-extraction technology-term filter.
    """
    idx, row = row_tuple
    chunk_id = row["chunk_id"]

    paragraph_text = str(row["text"])[:2000]

    def _call():
        return client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": "Extract public benefits. Be concise. Remove technology-specific language."},
                {"role": "user", "content": BENEFIT_EXTRACTION_PROMPT.format(text=paragraph_text)}
            ],
            max_completion_tokens=1500,
        )

    try:
        try:
            response = _call()
        except Exception:
            time.sleep(1.0)
            response = _call()  # one retry

        content = response.choices[0].message.content
        if content is None:
            _record_benefit_failure(chunk_id, "empty_response")
            return chunk_id, []

        content = content.strip()

        # Guard against model refusing due to missing text / page references
        if "I don't have the text" in content or "I don’t have the text" in content:
            _record_benefit_failure(chunk_id, "model_refused_no_text")
            return chunk_id, []

        if content == "NO_BENEFIT" or not content:
            return chunk_id, []

        benefits = [b.strip() for b in content.split("\n") if b.strip() and b.strip() != "NO_BENEFIT"]
        filtered = [b for b in benefits if not _benefit_phrase_contains_tech_term(b)]
        return chunk_id, filtered

    except Exception as e:
        _record_benefit_failure(chunk_id, f"exception: {type(e).__name__}: {e}")
        return chunk_id, []

# Cache file (benefits)
benefits_cache_file = CHECKPOINT_FOLDER / "extracted_benefits.json"

# Load cache if valid
if benefits_cache_file.exists():
    print("Found cached benefit extractions...")
    with open(benefits_cache_file) as f:
        cached_benefits = json.load(f)

    cached_ids = set(cached_benefits.keys())
    current_ids = set(chunks_df["chunk_id"].tolist())

    if cached_ids == current_ids:
        print(f"Using cached extractions for {len(cached_benefits)} paragraphs")
        all_benefits = cached_benefits
    else:
        print("Cache mismatch - will re-extract benefits")
        all_benefits = None
else:
    all_benefits = None

if all_benefits is None:
    all_benefits = {}

    # Metadata lookup
    chunk_metadata = chunks_df.set_index("chunk_id")[["technology", "year", "source_file"]].to_dict("index")

    MAX_WORKERS = 10
    rows = list(chunks_df.iterrows())

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(extract_benefits_from_paragraph, row): row[1]["chunk_id"] for row in rows}

        for future in tqdm(as_completed(futures), total=len(futures), desc="Extracting benefits"):
            chunk_id, benefits = future.result()
            meta = chunk_metadata[chunk_id]
            all_benefits[chunk_id] = {
                "benefits": benefits,
                "technology": meta["technology"],
                "year": int(meta["year"]) if pd.notna(meta["year"]) else None,
                "source_file": meta["source_file"]
            }

    with open(benefits_cache_file, "w") as f:
        json.dump(all_benefits, f, indent=2)

    show_complete(f"Extracted benefits from {len(all_benefits)} paragraphs")

if _benefit_extraction_failures:
    failure_log_path = OUTPUT_FOLDER / "benefit_extraction_failures.csv"
    pd.DataFrame(_benefit_extraction_failures, columns=["chunk_id", "reason"]).to_csv(failure_log_path, index=False)
    failure_rate = len(_benefit_extraction_failures) / max(1, len(chunks_df))
    show_warning(
        f"Benefit extraction had {len(_benefit_extraction_failures)} failures "
        f"({failure_rate:.1%} of chunks). See {failure_log_path}."
    )
    if failure_rate > 0.05:
        raise RuntimeError(
            f"Benefit extraction failure rate ({failure_rate:.1%}) exceeds 5% threshold. "
            f"Inspect {failure_log_path} before proceeding."
        )

# Flatten to individual benefit rows
benefit_rows = []
for chunk_id, data in all_benefits.items():
    for benefit in data["benefits"]:
        benefit_rows.append({
            "chunk_id": chunk_id,
            "benefit": benefit,
            "technology": data["technology"],
            "year": data["year"],
            "source_file": data["source_file"]
        })

benefits_df = pd.DataFrame(benefit_rows)
benefits_df["benefit_id"] = [f"benefit_{i}" for i in range(len(benefits_df))]

print(f"\nExtraction Summary:")
print(f"  Paragraphs processed: {len(all_benefits)}")
print(f"  Total benefit phrases: {len(benefits_df)}")
print(f"  Avg benefits per paragraph: {len(benefits_df) / max(1, len(all_benefits)):.2f}")
print(f"  Paragraphs with no benefits: {sum(1 for d in all_benefits.values() if len(d['benefits']) == 0)}")

benefits_df.to_csv(OUTPUT_FOLDER / "extracted_benefits.csv", index=False)


In [ ]:
# Make sure benefits_df knows the technology of each paragraph (safe merge)

# Prefer chunks_df's 'technology' if 'technology_meta' doesn't exist there
tech_col = "technology_meta" if "technology_meta" in chunks_df.columns else "technology"

benefits_df = benefits_df.merge(
    chunks_df[["chunk_id", tech_col]].rename(columns={tech_col: "technology_meta"}),
    on="chunk_id",
    how="left"
)

# If benefits_df already has 'technology' and it's missing where technology_meta exists, fill it
if "technology" in benefits_df.columns:
    benefits_df["technology"] = benefits_df["technology"].fillna(benefits_df["technology_meta"])
else:
    benefits_df["technology"] = benefits_df["technology_meta"]

In [ ]:
# @title Inspect sample benefit extractions

import pandas as pd

print("Sample extracted benefits by technology:")
print("(Checking that technology-specific language has been removed)\n")

# Identify the text column in benefits_df
candidate_cols = ["benefit", "benefit_phrase", "phrase", "extracted_phrase", "text"]
phrase_col = next((c for c in candidate_cols if c in benefits_df.columns), None)

if phrase_col is None:
    raise KeyError(
        f"Couldn't find a benefit text column in benefits_df. "
        f"Looked for {candidate_cols}. Available columns: {list(benefits_df.columns)}"
    )

# Identify technology column (you appear to have technology_meta)
tech_col = "technology_meta" if "technology_meta" in benefits_df.columns else (
    "technology" if "technology" in benefits_df.columns else None
)

if tech_col is None:
    raise KeyError(
        f"Couldn't find a technology column in benefits_df. "
        f"Available columns: {list(benefits_df.columns)}"
    )

for tech in benefits_df[tech_col].dropna().unique()[:6]:
    tech_benefits = benefits_df[benefits_df[tech_col] == tech][phrase_col].head(8).tolist()
    print(f"\n{tech}:")
    for b in tech_benefits:
        print(f"  • {b}")

---
# SECTION 3B: Embedding and clustering (benefits)
---

In [ ]:
# @title Generate semantic embeddings for benefit phrases

# Reuse get_embeddings_batch if already defined in Part I
if "get_embeddings_batch" not in globals():
    def get_embeddings_batch(texts, model=EMBEDDING_MODEL):
        response = client.embeddings.create(input=texts, model=model)
        return np.array([item.embedding for item in response.data])

embeddings_file = CHECKPOINT_FOLDER / "benefit_embeddings.npy"
benefit_ids_file = CHECKPOINT_FOLDER / "benefit_ids.json"

if embeddings_file.exists() and benefit_ids_file.exists():
    print("Found cached embeddings...")
    benefit_embeddings = np.load(embeddings_file)
    with open(benefit_ids_file) as f:
        cached_benefit_ids = json.load(f)

    if cached_benefit_ids == benefits_df["benefit_id"].tolist():
        print(f"Using cached embeddings: {benefit_embeddings.shape}")
    else:
        print("Cache mismatch - regenerating")
        benefit_embeddings = None
else:
    benefit_embeddings = None

if benefit_embeddings is None:
    show_status(f"Generating embeddings for {len(benefits_df)} benefit phrases...")

    texts = benefits_df["benefit"].tolist()
    all_embeddings = []
    failed_batch_starts = []

    for i in tqdm(range(0, len(texts), EMBEDDING_BATCH_SIZE), desc="Embedding"):
        batch = texts[i:i+EMBEDDING_BATCH_SIZE]
        try:
            batch_embeddings = get_embeddings_batch(batch)
            all_embeddings.append(batch_embeddings)
        except Exception as e:
            print(f"Error on batch {i}: {e}")
            failed_batch_starts.append(i)
        time.sleep(0.1)

    if failed_batch_starts:
        raise RuntimeError(
            f"{len(failed_batch_starts)} benefit embedding batch(es) failed: "
            f"starting indices {failed_batch_starts}. "
            f"Re-run this cell after investigating; the cache will be regenerated."
        )

    benefit_embeddings = np.vstack(all_embeddings) if all_embeddings else np.empty((0, 0))
    np.save(embeddings_file, benefit_embeddings)
    with open(benefit_ids_file, "w") as f:
        json.dump(benefits_df["benefit_id"].tolist(), f)

    show_complete(f"Generated embeddings: {benefit_embeddings.shape}")

assert benefit_embeddings.shape[0] == len(benefits_df), (
    f"Benefit embeddings count ({benefit_embeddings.shape[0]}) does not match "
    f"benefits_df rows ({len(benefits_df)}). Delete {embeddings_file} and re-run."
)

if benefit_embeddings.size:
    print(f"Embedding dimensions: {benefit_embeddings.shape[1]}")
else:
    print("No embeddings generated (benefits_df empty).")


In [ ]:
# @title Cluster benefit phrase embeddings

N_BENEFIT_CLUSTERS = globals().get("N_BENEFIT_CLUSTERS", N_CONCERN_CLUSTERS)

show_status(f"Clustering into {N_BENEFIT_CLUSTERS} benefit groups...")

benefit_embeddings_normalized = normalize(benefit_embeddings)

benefit_kmeans = KMeans(
    n_clusters=N_BENEFIT_CLUSTERS,
    random_state=RANDOM_SEED,
    n_init=10,
    max_iter=300
)
benefit_cluster_assignments = benefit_kmeans.fit_predict(benefit_embeddings_normalized)

benefit_centroids = benefit_kmeans.cluster_centers_
benefit_centroids_normalized = normalize(benefit_centroids)

assert len(benefit_cluster_assignments) == len(benefits_df), (
    f"Benefit cluster assignment length ({len(benefit_cluster_assignments)}) "
    f"does not match benefits_df rows ({len(benefits_df)})."
)

# Add cluster assignment to benefits dataframe
benefits_df["cluster_id"] = benefit_cluster_assignments

# SOFT MEMBERSHIP: cosine similarity to each centroid
show_status("Computing soft membership scores...")
benefit_soft_membership = cosine_similarity(benefit_embeddings_normalized, benefit_centroids_normalized)

print(f"\nClustering Results:")
print(f"  Clusters: {N_BENEFIT_CLUSTERS}")
print(f"  Benefit phrases: {len(benefits_df)}")
print(f"  Soft membership matrix: {benefit_soft_membership.shape}")

benefit_cluster_sizes = np.bincount(benefit_cluster_assignments)
print(f"  Cluster sizes: min={benefit_cluster_sizes.min()}, max={benefit_cluster_sizes.max()}, median={np.median(benefit_cluster_sizes):.0f}")

np.save(CHECKPOINT_FOLDER / "benefit_soft_membership.npy", benefit_soft_membership)
np.save(CHECKPOINT_FOLDER / "benefit_cluster_centroids.npy", benefit_centroids_normalized)

show_complete("Benefit clustering complete")


In [ ]:
# @title Characterise benefit clusters by technology distribution

from scipy.stats import entropy as scipy_entropy

show_status("Analysing benefit cluster composition...")

benefit_cluster_entropy = {}
benefit_cluster_tech_dist = {}

technologies = benefits_df['technology_meta'].unique()
n_techs = len(technologies)

for cluster_id in range(N_BENEFIT_CLUSTERS):
    cluster_mask = benefits_df['cluster_id'] == cluster_id
    cluster_data = benefits_df[cluster_mask]

    if len(cluster_data) == 0:
        benefit_cluster_entropy[cluster_id] = 0
        benefit_cluster_tech_dist[cluster_id] = {}
        continue

    # Technology distribution
    tech_counts = cluster_data['technology_meta'].value_counts()
    tech_probs = tech_counts / tech_counts.sum()

    # Entropy (higher = more cross-cutting)
    benefit_cluster_entropy[cluster_id] = scipy_entropy(tech_probs)
    benefit_cluster_tech_dist[cluster_id] = tech_counts.to_dict()

# Normalize entropy to 0-1 scale (max entropy = log(n_techs))
max_entropy = np.log(n_techs)
normalized_entropy_benefits = {k: v / max_entropy for k, v in benefit_cluster_entropy.items()}

# Classify clusters
CROSS_CUTTING_THRESHOLD = 0.5  # Entropy > 50% of max = cross-cutting

cross_cutting_clusters_benefits = [k for k, v in normalized_entropy_benefits.items() if v >= CROSS_CUTTING_THRESHOLD]
tech_specific_clusters_benefits = [k for k, v in normalized_entropy_benefits.items() if v < CROSS_CUTTING_THRESHOLD]

print(f"\nBenefit Cluster Classification:")
print(f"  Cross-cutting clusters (entropy >= {CROSS_CUTTING_THRESHOLD}): {len(cross_cutting_clusters_benefits)}")
print(f"  Technology-specific clusters: {len(tech_specific_clusters_benefits)}")
print(f"  Ratio: {len(cross_cutting_clusters_benefits)/N_BENEFIT_CLUSTERS*100:.1f}% cross-cutting")

# Visualise entropy distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Histogram of entropy
axes[0].hist(list(normalized_entropy_benefits.values()), bins=20, color='steelblue', edgecolor='white')
axes[0].axvline(CROSS_CUTTING_THRESHOLD, color='red', linestyle='--', label=f'Threshold ({CROSS_CUTTING_THRESHOLD})')
axes[0].set_xlabel('Normalized Entropy')
axes[0].set_ylabel('Number of Clusters')
axes[0].set_title('Benefit Cluster Entropy Distribution\n(Higher = More Cross-Cutting)')
axes[0].legend()

# Entropy vs cluster size
sizes = [int((benefits_df["cluster_id"] == i).sum()) for i in range(N_BENEFIT_CLUSTERS)]
entropies = [normalized_entropy_benefits[i] for i in range(N_BENEFIT_CLUSTERS)]
colors = ['green' if e >= CROSS_CUTTING_THRESHOLD else 'orange' for e in entropies]
axes[1].scatter(sizes, entropies, c=colors, alpha=0.6)
axes[1].axhline(CROSS_CUTTING_THRESHOLD, color='red', linestyle='--')
axes[1].set_xlabel('Cluster Size')
axes[1].set_ylabel('Normalized Entropy')
axes[1].set_title('Cross-Cutting (green) vs Technology-Specific (orange)')

plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "benefit_cluster_entropy_analysis.png", dpi=150)
plt.show()

show_complete("Benefit cluster composition analysis complete")


In [ ]:
# --- Define cross-cutting helpers for Benefits (Section 6) ---

CROSS_CUTTING_ENTROPY_THRESHOLD = globals().get("CROSS_CUTTING_THRESHOLD", 0.5)

if "cross_cutting_clusters_benefits" not in globals():
    raise NameError("cross_cutting_clusters_benefits not found. Run the benefit entropy/classification cell first.")

cross_cutting_cluster_ids_benefits = set(cross_cutting_clusters_benefits)

label_map = {}
if "benefit_summary_df" in globals() and {"cluster_id", "label"}.issubset(benefit_summary_df.columns):
    label_map = dict(zip(benefit_summary_df["cluster_id"], benefit_summary_df["label"]))

cross_cutting_labels_benefits = {
    cid: label_map.get(cid, f"Cluster {cid}") for cid in cross_cutting_cluster_ids_benefits
}


---
# SECTION 4B: Cluster interpretation and labelling (benefits)
---

In [ ]:
# @title Extract exemplar benefit phrases per cluster

show_status("Extracting exemplar benefits...")

N_BENEFIT_CLUSTERS = globals().get("N_BENEFIT_CLUSTERS", N_CONCERN_CLUSTERS)
N_EXEMPLARS = 8
benefit_cluster_exemplars = {}

assert "cluster_id" in benefits_df.columns, "benefits_df has no cluster_id column. Run cell 58 first."
assert benefits_df["cluster_id"].max() < N_BENEFIT_CLUSTERS, (
    f"benefits_df.cluster_id contains values >= {N_BENEFIT_CLUSTERS}. "
    "This usually means stale cluster IDs from a different clustering run; re-run cell 58."
)

for cluster_id in range(N_BENEFIT_CLUSTERS):
    cluster_mask = benefits_df["cluster_id"] == cluster_id
    cluster_benefits = benefits_df[cluster_mask]
    cluster_embs = benefit_embeddings_normalized[cluster_mask]

    if len(cluster_benefits) == 0:
        continue

    centroid = benefit_centroids_normalized[cluster_id]
    similarities = cosine_similarity(cluster_embs, centroid.reshape(1, -1)).flatten()
    top_indices = np.argsort(similarities)[-N_EXEMPLARS:][::-1]

    exemplars = []
    for i in top_indices:
        row = cluster_benefits.iloc[i]
        exemplars.append({
            "benefit": row["benefit"],
            "technology": row["technology"],
            "year": int(row["year"]) if pd.notna(row["year"]) else None,
            "similarity": float(similarities[i])
        })

    tech_dist = cluster_benefits["technology"].value_counts().head(3).to_dict()

    benefit_cluster_exemplars[cluster_id] = {
        "size": len(cluster_benefits),
        "entropy": normalized_entropy_benefits[cluster_id],
        "is_cross_cutting": cluster_id in cross_cutting_clusters_benefits,
        "top_technologies": tech_dist,
        "exemplars": exemplars
    }

with open(OUTPUT_FOLDER / "benefit_cluster_exemplars.json", "w") as f:
    json.dump(benefit_cluster_exemplars, f, indent=2, default=str)

show_complete(f"Extracted exemplars for {len(benefit_cluster_exemplars)} clusters")


In [ ]:
# @title Assign descriptive labels to benefit clusters

MIN_CLUSTER_LABEL_SUCCESS_RATE = 0.90

def label_benefit_cluster(cluster_id, exemplars, is_cross_cutting):
    """    function of the same name.
    fix for label_cluster() in cell 24."""
    exemplar_texts = "\n".join([f"- {ex['benefit']}" for ex in exemplars[:8]])

    prompt = f"""Analyze this cluster of public benefits from UK dialogue reports.

Benefit phrases in this cluster:
{exemplar_texts}

Provide:
1. SHORT LABEL (3-6 words) capturing the core benefit theme.
   Use neutral, generic language; do NOT prefix the label with a specific
   technology (e.g. write "Improved diagnosis", not "AI-driven improved diagnosis").
2. DESCRIPTION (1-2 sentences)
3. KEY TERMS (3-5 representative words/phrases)

Return JSON only:
{{"label": "...", "description": "...", "key_terms": ["...", "..."]}}"""

    try:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": "Expert qualitative researcher. Return only valid JSON."},
                {"role": "user", "content": prompt}
            ],
            max_completion_tokens=1000,
        )
        content = response.choices[0].message.content
        if content is None or not content.strip():
            return {"label": f"Cluster {cluster_id}", "description": "", "key_terms": [], "success": False}

        content = content.strip()
        if "```" in content:
            parts = content.split("```")
            for part in parts:
                if part.startswith("json"):
                    content = part[4:].strip()
                    break
                elif part.strip().startswith("{"):
                    content = part.strip()
                    break

        result = json.loads(content)
        result["success"] = True
        return result
    except Exception as e:
        return {"label": f"Cluster {cluster_id}", "description": "", "key_terms": [], "success": False, "error": str(e)}

N_BENEFIT_CLUSTERS = globals().get("N_BENEFIT_CLUSTERS", N_CONCERN_CLUSTERS)

# Ensure exemplars dict exists
if "benefit_cluster_exemplars" not in globals():
    raise NameError("benefit_cluster_exemplars not found. Run the exemplar extraction cell first.")

# Cache for benefit labels
benefit_labels_file = OUTPUT_FOLDER / "benefit_cluster_labels.json"

benefit_cluster_labels_dict = None
if benefit_labels_file.exists():
    print("Found cached benefit cluster labels...")
    with open(benefit_labels_file) as f:
        benefit_cluster_labels_dict = json.load(f)

    if len(benefit_cluster_labels_dict) == N_BENEFIT_CLUSTERS:
        print(f"Using cached labels for {len(benefit_cluster_labels_dict)} clusters")
    else:
        print("Cache incomplete - regenerating")
        benefit_cluster_labels_dict = None

if benefit_cluster_labels_dict is None:
    show_status(f"Generating labels for {len(benefit_cluster_exemplars)} clusters...")

    benefit_cluster_labels_dict = {}
    failed_count = 0

    for cluster_id in tqdm(benefit_cluster_exemplars.keys(), desc="Labeling"):
        data = benefit_cluster_exemplars[cluster_id]
        result = label_benefit_cluster(cluster_id, data["exemplars"], data["is_cross_cutting"])
        benefit_cluster_labels_dict[str(cluster_id)] = result

        if not result.get("success", False):
            failed_count += 1

        time.sleep(0.3)

    with open(benefit_labels_file, "w") as f:
        json.dump(benefit_cluster_labels_dict, f, indent=2)

    show_complete(f"Labeling complete: {len(benefit_cluster_labels_dict) - failed_count} successful, {failed_count} failed")

_label_success = sum(1 for v in benefit_cluster_labels_dict.values() if v.get("success", False))
_label_total = max(1, len(benefit_cluster_labels_dict))
_label_success_rate = _label_success / _label_total
print(f"Benefit cluster-label success rate: {_label_success_rate:.1%} ({_label_success}/{_label_total})")
if _label_success_rate < MIN_CLUSTER_LABEL_SUCCESS_RATE:
    raise RuntimeError(
        f"Benefit cluster-label success rate ({_label_success_rate:.1%}) below threshold "
        f"({MIN_CLUSTER_LABEL_SUCCESS_RATE:.0%})."
    )

# Create label list (optional downstream use)
benefit_cluster_labels_list = [
    benefit_cluster_labels_dict.get(str(i), {}).get("label", f"Cluster {i}")
    for i in range(N_BENEFIT_CLUSTERS)
]

# Summary table
print("\nTop 15 Benefit Clusters by Size:")
cluster_summary = []
for cluster_id, data in benefit_cluster_exemplars.items():
    label = benefit_cluster_labels_dict.get(str(cluster_id), {}).get("label", f"Cluster {cluster_id}")
    cluster_summary.append({
        "cluster_id": cluster_id,
        "label": label,
        "size": data["size"],
        "entropy": data["entropy"],
        "type": "Cross-cutting" if data["is_cross_cutting"] else "Tech-specific"
    })

benefit_summary_df = pd.DataFrame(cluster_summary).sort_values("size", ascending=False)
print(benefit_summary_df[["cluster_id", "label", "size", "type"]].head(15).to_string(index=False))

benefit_summary_df.to_csv(OUTPUT_FOLDER / "benefit_cluster_summary.csv", index=False)


---
# SECTION 5B: Framing lens analysis (benefits)
---

In [ ]:
# @title Identify benefit framing lenses

show_status("Generating benefit framing lens suggestions...")

N_BENEFIT_CLUSTERS = globals().get("N_BENEFIT_CLUSTERS", N_CONCERN_CLUSTERS)

# Prepare cluster summary for GPT
cluster_info = []
for cluster_id, data in benefit_cluster_exemplars.items():
    label = benefit_cluster_labels_dict.get(str(cluster_id), {}).get('label', f'Cluster {cluster_id}')
    description = benefit_cluster_labels_dict.get(str(cluster_id), {}).get('description', '')
    cluster_type = 'cross-cutting' if data['is_cross_cutting'] else 'tech-specific'
    cluster_info.append(f"{cluster_id}. {label} ({cluster_type}, n={data['size']}): {description}")

cluster_summary_text = "\n".join(cluster_info)

lens_prompt = f"""Analyze these {N_BENEFIT_CLUSTERS} benefit clusters from UK public dialogue reports.

Clusters:
{cluster_summary_text}

Group these clusters into 8-12 higher-level FRAMING LENSES that capture how publics frame their benefits.

For each lens provide:
1. Name (2-4 words)
2. Description (1 sentence)
3. List of cluster IDs that belong to this lens

A cluster can belong to multiple lenses if appropriate.
Ensure all clusters are assigned to at least one lens.

Return JSON:
{{"framing_lenses": [
  {{"name": "...", "description": "...", "suggested_clusters": [0, 1, 2...]}},
  ...
]}}"""

try:
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": "Expert in public engagement and discourse analysis. Return only valid JSON."},
            {"role": "user", "content": lens_prompt}
        ],
        max_completion_tokens=8000,
    )

    content = response.choices[0].message.content
    if '```' in content:
        parts = content.split('```')
        for part in parts:
            if part.startswith('json'):
                content = part[4:].strip()
                break
            elif part.strip().startswith('{'):
                content = part.strip()
                break

    suggested_benefit_lenses = json.loads(content)

    print("Suggested Benefit Framing Lenses:\n")
    for lens in suggested_benefit_lenses['framing_lenses']:
        n_clusters = len(lens['suggested_clusters'])
        print(f"  {lens['name']}")
        print(f"    {lens['description']}")
        print(f"    Clusters: {lens['suggested_clusters'][:8]}...\n")

    show_complete(f"Generated {len(suggested_benefit_lenses['framing_lenses'])} benefit framing lenses")

except Exception as e:
    print(f"Error generating lenses: {e}")
    suggested_benefit_lenses = {'framing_lenses': []}


In [ ]:
# @title Map benefit clusters to framing lenses

BENEFIT_FRAMING_LENS_MAPPINGS = {}
for lens in suggested_benefit_lenses['framing_lenses']:
    BENEFIT_FRAMING_LENS_MAPPINGS[lens['name']] = {
        'description': lens['description'],
        'cluster_ids': lens['suggested_clusters']
    }

with open(OUTPUT_FOLDER / "benefit_framing_lens_mappings.json", 'w') as f:
    json.dump(BENEFIT_FRAMING_LENS_MAPPINGS, f, indent=2)

# Map clusters to lenses (benefit-specific)
benefit_cluster_to_lens = {}
for lens_name, data in BENEFIT_FRAMING_LENS_MAPPINGS.items():
    for cluster_id in data['cluster_ids']:
        if cluster_id not in benefit_cluster_to_lens:
            benefit_cluster_to_lens[cluster_id] = []
        benefit_cluster_to_lens[cluster_id].append(lens_name)

all_cluster_ids = set(range(N_BENEFIT_CLUSTERS))
covered_cluster_ids = set(benefit_cluster_to_lens.keys())
uncovered = all_cluster_ids - covered_cluster_ids
if uncovered:
    print("Uncovered benefit cluster IDs:", sorted(uncovered)[:30])
    raise RuntimeError(
        f"{len(uncovered)} of {N_BENEFIT_CLUSTERS} benefit clusters are not assigned to any "
        f"framing lens. Re-run the framing-lens identification cell or assign manually."
    )

print("Benefit Framing Lens Mappings:")
for lens_name, data in BENEFIT_FRAMING_LENS_MAPPINGS.items():
    n_clusters = len(data['cluster_ids'])
    lens_mask = benefits_df['cluster_id'].isin(data['cluster_ids'])
    n_benefits = lens_mask.sum()
    print(f"  {lens_name}: {n_clusters} clusters, {n_benefits} benefits")


In [ ]:
# @title Compute benefit framing lens prevalence

show_status("Computing benefit framing lens prevalence...")

lens_prevalence = {}
for lens_name, data in BENEFIT_FRAMING_LENS_MAPPINGS.items():
    lens_mask = benefits_df['cluster_id'].isin(data['cluster_ids'])
    lens_prevalence[lens_name] = lens_mask.sum()

lens_prev_df = pd.DataFrame([
    {'Framing Lens': k, 'Benefits': v}
    for k, v in lens_prevalence.items()
]).sort_values('Benefits', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(lens_prev_df['Framing Lens'], lens_prev_df['Benefits'], color='steelblue')
ax.set_xlabel('Number of Benefit Phrases')
ax.set_title('Benefit Framing Lens Prevalence Across All Documents')

for bar, val in zip(bars, lens_prev_df['Benefits']):
    ax.text(val + 10, bar.get_y() + bar.get_height()/2, f'{val:,}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "benefit_framing_lens_prevalence.png", dpi=150, bbox_inches='tight')
plt.show()

show_complete("Benefit framing lens prevalence computed")


In [ ]:
# @title Hierarchical structure of benefit clusters (dendrogram)
# Validates the framing-lens scheme by showing whether benefit clusters
# that the LLM grouped into the same lens also cluster together in a
# data-driven hierarchy. Strong correspondence supports the lens scheme;
# divergence is a finding worth reporting.

import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist

show_status("Computing benefit-cluster dendrogram...")

# L2-normalise benefit centroids (cosine distance via euclidean on normed vectors)
_norm = np.linalg.norm(benefit_centroids, axis=1, keepdims=True)
benefit_centroids_normed = benefit_centroids / np.clip(_norm, 1e-12, None)

# Pairwise cosine distance and Ward linkage
distances = pdist(benefit_centroids_normed, metric="cosine")
Z = linkage(distances, method="average")  # average linkage works well with cosine

# Build leaf labels using existing benefit cluster labels
benefit_cluster_label_lookup = {
    int(k): v.get("label", f"Cluster {k}")
    for k, v in benefit_cluster_labels_dict.items()
}
leaf_labels = [benefit_cluster_label_lookup.get(i, f"Cluster {i}")
               for i in range(len(benefit_centroids_normed))]

# Build cluster->lens lookup so we can colour leaves by lens
benefit_cluster_to_lens = {}
for lens_name, info in BENEFIT_FRAMING_LENS_MAPPINGS.items():
    for cid in info.get("cluster_ids", []):
        benefit_cluster_to_lens[int(cid)] = lens_name

# Assign a colour per lens
import matplotlib.cm as cm
lens_names_b = list(BENEFIT_FRAMING_LENS_MAPPINGS.keys())
lens_colours_b = {name: cm.tab20(i / max(len(lens_names_b), 1))
                  for i, name in enumerate(lens_names_b)}

# Plot
fig, ax = plt.subplots(figsize=(11, max(8, len(leaf_labels) * 0.18)))
dd = dendrogram(
    Z,
    labels=leaf_labels,
    orientation="left",
    leaf_font_size=8,
    color_threshold=0,
    above_threshold_color="grey",
    ax=ax,
)
# Recolour leaf labels by lens
ylabels = ax.get_ymajorticklabels()
for label in ylabels:
    cid_label = label.get_text()
    matched_cid = next((cid for cid, name in benefit_cluster_label_lookup.items()
                        if name == cid_label), None)
    if matched_cid is not None:
        lens = benefit_cluster_to_lens.get(matched_cid, None)
        if lens is not None:
            label.set_color(lens_colours_b.get(lens, "black"))

ax.set_xlabel("Cosine distance between benefit cluster centroids")
ax.set_title("Hierarchical structure of benefit clusters\n(leaf colour = LLM-assigned framing lens)")

# Legend
from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=col, label=name) for name, col in lens_colours_b.items()]
ax.legend(handles=legend_handles, loc="lower right", fontsize=7, title="Framing lens")

plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "benefit_cluster_dendrogram.png", dpi=200, bbox_inches="tight")
plt.show()

# Quantitative comparison: cut the dendrogram at the same number of branches
# as the lens scheme, then compute Adjusted Rand Index against the lens
# assignment. If high, the lenses are well-supported by the data structure.
from sklearn.metrics import adjusted_rand_score

n_lenses_b = len(BENEFIT_FRAMING_LENS_MAPPINGS)
data_driven_groups = fcluster(Z, t=n_lenses_b, criterion="maxclust")
lens_assignment = np.array([
    lens_names_b.index(benefit_cluster_to_lens.get(i, lens_names_b[0]))
    if benefit_cluster_to_lens.get(i) in lens_names_b else -1
    for i in range(len(benefit_centroids_normed))
])
mask = lens_assignment >= 0
ari_b = adjusted_rand_score(lens_assignment[mask], data_driven_groups[mask])
print(f"\nNumber of benefit lenses (LLM-derived): {n_lenses_b}")
print(f"Adjusted Rand Index between lens assignment and dendrogram cut at "
      f"{n_lenses_b} groups: {ari_b:.3f}")
print("Higher = stronger correspondence between data-driven hierarchy and "
      "LLM-derived lens scheme.")

show_complete("Benefit-cluster dendrogram complete.")

# SECTION 6B: Shared benefits, shared framings, and AI distinctiveness (benefits)
This section has three goals:

**6.1 Shared structure (all technologies, document‑weighted):** What benefits and framings recur across technologies?

**6.2 AI vs non‑AI baseline (technology‑weighted):** How does AI's profile differ from the average *non‑AI technology* (to avoid circularity, because AI documents are a large share of the corpus)?

**6.3 Distinctiveness:** Which benefits / framings are over‑ or under‑represented in AI relative to the non‑AI baseline?

**Important data contract:** all technology labels come exclusively from the metadata file (see `technology_meta`). LLM‑generated cluster descriptions must never be used as technology categories.


In [ ]:
# --- Section 6 prerequisites (canonical tech + cross-cutting clusters) ---

TECH_COL = "technology_meta"

# Ensure benefits_df has technology_meta
if TECH_COL not in benefits_df.columns:
    benefits_df = benefits_df.merge(
        chunks_df[["chunk_id", TECH_COL]],
        on="chunk_id",
        how="left"
    )

CROSS_CUTTING_ENTROPY_THRESHOLD = 0.5
cross_cutting_cluster_ids_benefits = {
    cid for cid, ent in normalized_entropy_benefits.items()
    if ent >= CROSS_CUTTING_ENTROPY_THRESHOLD
}


In [ ]:
# @title Shared benefits across technologies (document-weighted)

show_status("Computing shared benefit structure (all technologies)...")

TECH_COL = "technology_meta"

# Canonical list of technologies from metadata
technologies = sorted(benefits_df[TECH_COL].dropna().unique().tolist())

benefit_salience_rows = {}
for tech in technologies:
    tech_mask = benefits_df[TECH_COL] == tech
    tech_total = int(tech_mask.sum())
    if tech_total == 0:
        continue
    counts = benefits_df.loc[tech_mask, "cluster_id"].value_counts()
    benefit_salience_rows[tech] = (counts / tech_total).to_dict()

benefit_salience_df = pd.DataFrame(benefit_salience_rows).T.fillna(0)

# Document-weighted global prevalence of clusters
benefit_global_cluster_prevalence = benefits_df["cluster_id"].value_counts(normalize=True)

benefit_cluster_meta_df = (
    pd.DataFrame(
        {
            "cluster_id": list(benefit_cluster_entropy.keys()),
            "tech_entropy": [benefit_cluster_entropy[c] for c in benefit_cluster_entropy.keys()],
            "global_prevalence": [benefit_global_cluster_prevalence.get(c, 0) for c in benefit_cluster_entropy.keys()],
        }
    )
    .set_index("cluster_id")
    .sort_values(["global_prevalence", "tech_entropy"], ascending=False)
)

# Shared benefits = high prevalence + high entropy
shared_benefits = benefit_cluster_meta_df.head(20)

print("Top shared benefit clusters (high prevalence + cross-technology spread):")
display(shared_benefits.head(12))

plt.figure(figsize=(10, 5))
shared_benefits.sort_values("global_prevalence")["global_prevalence"].plot(kind="barh", legend=False)
plt.title("Shared benefits across technologies (top 20 clusters by prevalence x spread)")
plt.xlabel("Share of all extracted benefit phrases")
plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "shared_benefits_top20.png")
plt.show()

shared_benefits.reset_index().to_csv(OUTPUT_FOLDER / "shared_benefits_top20.csv", index=False)

show_complete("Shared benefits computed")


In [ ]:
# @title Shared benefit framings across technologies (document-weighted)

from scipy.stats import entropy as scipy_entropy

show_status("Computing shared benefit framing structure (all technologies)...")

TECH_COL = 'technology_meta'
technologies = sorted(benefits_df[TECH_COL].dropna().unique().tolist())

# Lens prevalence overall + entropy across technologies
benefit_lens_stats = []
for lens_name, data in BENEFIT_FRAMING_LENS_MAPPINGS.items():
    lens_clusters = set(data['cluster_ids'])
    lens_mask = benefits_df['cluster_id'].isin(lens_clusters)
    overall_prev = lens_mask.mean()

    tech_counts = benefits_df.loc[lens_mask, TECH_COL].value_counts()
    tech_probs = (tech_counts / tech_counts.sum()).values if tech_counts.sum() > 0 else np.array([])
    lens_entropy = float(scipy_entropy(tech_probs)) if len(tech_probs) > 1 else 0.0

    benefit_lens_stats.append({
        'framing_lens': lens_name,
        'overall_prevalence': float(overall_prev),
        'tech_entropy': lens_entropy,
        'n_benefits_in_lens': int(lens_mask.sum())
    })

benefit_lens_meta_df = (pd.DataFrame(benefit_lens_stats)
                        .sort_values(['overall_prevalence','tech_entropy'], ascending=False))

print("Shared benefit framings (high prevalence + cross-technology spread):")
display(benefit_lens_meta_df.head(12))

plt.figure(figsize=(7,5))
plt.scatter(benefit_lens_meta_df['tech_entropy'], benefit_lens_meta_df['overall_prevalence'])
for _, r in benefit_lens_meta_df.iterrows():
    plt.text(r['tech_entropy'], r['overall_prevalence'], r['framing_lens'], fontsize=8)
plt.xlabel("Entropy across technologies")
plt.ylabel("Share of all extracted benefits")
plt.title("Shared benefit framings across technologies")
plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "shared_benefit_framings_scatter.png")
plt.show()

benefit_lens_meta_df.to_csv(OUTPUT_FOLDER / "shared_benefit_framings.csv", index=False)
show_complete("Shared benefit framings computed")


In [ ]:
# @title Compare AI and non-AI dialogues by framing lens (benefits)

show_status("Computing AI vs non-AI benefit framing profile (technology-weighted baseline)...")

TECH_COL = 'technology_meta'
technologies = sorted(benefits_df[TECH_COL].dropna().unique().tolist())
non_ai_techs = [t for t in technologies if t != 'AI']

# Lens salience per technology (within-technology proportions)
benefit_lens_salience_by_tech = {}
for tech in technologies:
    tech_mask = benefits_df[TECH_COL] == tech
    tech_total = tech_mask.sum()
    if tech_total == 0:
        continue
    benefit_lens_salience_by_tech[tech] = {}
    for lens_name, data in BENEFIT_FRAMING_LENS_MAPPINGS.items():
        lens_mask = benefits_df['cluster_id'].isin(data['cluster_ids'])
        benefit_lens_salience_by_tech[tech][lens_name] = float((tech_mask & lens_mask).sum() / tech_total)

benefit_lens_salience_df = pd.DataFrame(benefit_lens_salience_by_tech).T.fillna(0)

# Technology-weighted non-AI baseline (each non-AI technology counts equally)
benefit_non_ai_avg = benefit_lens_salience_df.loc[non_ai_techs].mean(axis=0) if len(non_ai_techs) else pd.Series(dtype=float)

# Plot radar: AI vs non-AI avg
if 'AI' in benefit_lens_salience_df.index and len(benefit_non_ai_avg) > 0:
    categories = list(benefit_non_ai_avg.index)
    ai_vals = benefit_lens_salience_df.loc['AI', categories].tolist()
    base_vals = benefit_non_ai_avg[categories].tolist()

    # Close the loop
    categories_loop = categories + [categories[0]]
    ai_loop = ai_vals + [ai_vals[0]]
    base_loop = base_vals + [base_vals[0]]

    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(r=ai_loop, theta=categories_loop, fill='toself', name='AI'))
    fig.add_trace(go.Scatterpolar(r=base_loop, theta=categories_loop, fill='toself', name='Non‑AI average (tech‑weighted)'))
    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, max(ai_loop+base_loop)*1.1])),
        title="AI benefit framing profile vs non‑AI baseline (technology‑weighted)"
    )
    fig.write_html(OUTPUT_FOLDER / "benefit_ai_vs_nonai_framing_radar.html")
# v18: also save a static PNG copy for paper use (requires kaleido package)
try:
    fig.write_image(OUTPUT_FOLDER / "benefit_ai_vs_nonai_framing_radar.png", width=900, height=700, scale=2)
except Exception as _e:
    print(f"Note: PNG export failed ({_e}); HTML version still saved.")
    print("If you need PNG, install kaleido: !pip install -q kaleido")
    fig.show()

# Save table
benefit_lens_salience_df.to_csv(OUTPUT_FOLDER / "benefit_lens_salience_by_technology_meta.csv")
benefit_non_ai_avg.to_frame('non_ai_avg_tech_weighted').to_csv(OUTPUT_FOLDER / "benefit_non_ai_avg_framing_profile.csv")
show_complete("AI vs non-AI benefit framing profile computed")


In [ ]:
# @title Compare AI and non-AI dialogues on cross-cutting benefit clusters

show_status("Computing AI vs non-AI benefit profile on cross-cutting clusters...")

TECH_COL = 'technology_meta'
technologies = sorted(benefits_df[TECH_COL].dropna().unique().tolist())
non_ai_techs = [t for t in technologies if t != 'AI']

# Use cross_cutting_labels_benefits from cell 60 (derived from entropy threshold)
cross_cutting_cluster_ids_benefits = set(cross_cutting_labels_benefits.keys())

# Technology profiles over cross-cutting clusters
benefit_profiles = {}
for tech in technologies:
    tech_mask = benefits_df[TECH_COL] == tech
    tech_total = tech_mask.sum()
    if tech_total == 0:
        continue
    counts = benefits_df.loc[tech_mask & benefits_df['cluster_id'].isin(cross_cutting_cluster_ids_benefits), 'cluster_id'].value_counts()
    # normalise over all benefits for that tech (so "attention share" to cross-cutting clusters)
    benefit_profiles[tech] = (counts / tech_total).to_dict()

benefit_profiles_df = pd.DataFrame(benefit_profiles).T.fillna(0)

benefit_non_ai_avg_cluster = benefit_profiles_df.loc[non_ai_techs].mean(axis=0) if len(non_ai_techs) else pd.Series(dtype=float)

benefit_profiles_df.to_csv(OUTPUT_FOLDER / "benefit_cross_cutting_cluster_profiles_by_tech.csv")
benefit_non_ai_avg_cluster.to_frame('non_ai_avg_tech_weighted').to_csv(OUTPUT_FOLDER / "benefit_cross_cutting_cluster_profile_non_ai_avg.csv")

show_complete("Cross-cutting benefit profiles computed")


In [ ]:
# --- Build BENEFIT_CLUSTER_LABELS dict for plots (cluster_id -> label) ---
if "benefit_summary_df" in globals() and {"cluster_id", "label"}.issubset(benefit_summary_df.columns):
    BENEFIT_CLUSTER_LABELS = dict(zip(benefit_summary_df["cluster_id"], benefit_summary_df["label"]))
else:
    BENEFIT_CLUSTER_LABELS = {}


In [ ]:
# @title Visualise AI benefit fingerprint (AI vs non-AI)

TECH_COL = 'technology_meta'

if 'AI' in benefit_profiles_df.index and len(benefit_non_ai_avg_cluster) > 0:
    show_status("Creating AI benefit fingerprint radar (AI vs non-AI)...")

    diff = (benefit_profiles_df.loc['AI'] - benefit_non_ai_avg_cluster).sort_values()
    top_low = diff.head(5).index.tolist()
    top_high = diff.tail(7).index.tolist()
    selected = top_low + top_high

    # Human-friendly labels
    def pretty_cluster(cid):
        return BENEFIT_CLUSTER_LABELS.get(cid, f"Cluster {cid}")

    categories = [pretty_cluster(c) for c in selected]
    ai_vals = benefit_profiles_df.loc['AI', selected].tolist()
    base_vals = benefit_non_ai_avg_cluster[selected].tolist()

    categories_loop = categories + [categories[0]]
    ai_loop = ai_vals + [ai_vals[0]]
    base_loop = base_vals + [base_vals[0]]

    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(r=ai_loop, theta=categories_loop, fill='toself', name='AI'))
    fig.add_trace(go.Scatterpolar(r=base_loop, theta=categories_loop, fill='toself', name='Non‑AI average (tech‑weighted)'))
    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, max(ai_loop+base_loop)*1.1])),
        title="AI benefit fingerprint vs non‑AI baseline (cross‑cutting clusters)"
    )
    fig.write_html(OUTPUT_FOLDER / "benefit_ai_vs_nonai_radar.html")
# v18: also save a static PNG copy for paper use (requires kaleido package)
try:
    fig.write_image(OUTPUT_FOLDER / "benefit_ai_vs_nonai_radar.png", width=900, height=700, scale=2)
except Exception as _e:
    print(f"Note: PNG export failed ({_e}); HTML version still saved.")
    print("If you need PNG, install kaleido: !pip install -q kaleido")
    fig.show()

show_complete("AI benefit fingerprint radar created")


In [ ]:
# @title Identify distinctive AI benefits and framings

show_status("Computing distinctive benefits and framings for AI...")

TECH_COL = 'technology_meta'
non_ai_techs = [t for t in technologies if t != 'AI']

# Distinctive benefit clusters (all clusters, tech-weighted baseline)
all_cluster_profiles = benefit_salience_df  # rows=tech, cols=cluster_id

if 'AI' in all_cluster_profiles.index and len(non_ai_techs) > 0:
    non_ai_avg_all = all_cluster_profiles.loc[non_ai_techs].mean(axis=0)
    ai_vs_base = (all_cluster_profiles.loc['AI'] - non_ai_avg_all).sort_values()

    most_under = ai_vs_base.head(10)
    most_over = ai_vs_base.tail(10)

    def label(cid):
        return BENEFIT_CLUSTER_LABELS.get(cid, f"Cluster {cid}")

    distinctive_benefits = pd.DataFrame({
        'cluster_id': list(most_under.index) + list(most_over.index),
        'cluster_label': [label(c) for c in list(most_under.index) + list(most_over.index)],
        'ai_minus_nonai': list(most_under.values) + list(most_over.values),
        'ai_share': [float(all_cluster_profiles.loc['AI', c]) for c in list(most_under.index) + list(most_over.index)],
        'nonai_share': [float(non_ai_avg_all[c]) for c in list(most_under.index) + list(most_over.index)],
    }).sort_values('ai_minus_nonai')

    display(distinctive_benefits)
    distinctive_benefits.to_csv(OUTPUT_FOLDER / "ai_distinctive_benefits.csv", index=False)

# Distinctive framings (lens level) — use benefit_lens_salience_df
if 'AI' in benefit_lens_salience_df.index and len(non_ai_techs) > 0:
    non_ai_avg_lens = benefit_lens_salience_df.loc[non_ai_techs].mean(axis=0)
    lens_diff = (benefit_lens_salience_df.loc['AI'] - non_ai_avg_lens).sort_values()

    df = pd.DataFrame({
        'framing_lens': lens_diff.index,
        'ai_minus_nonai': lens_diff.values,
        'ai_share': benefit_lens_salience_df.loc['AI', lens_diff.index].values,
        'nonai_share': non_ai_avg_lens[lens_diff.index].values
    }).sort_values('ai_minus_nonai')

    display(df.head(10))
    display(df.tail(10))
    df.to_csv(OUTPUT_FOLDER / "benefit_ai_distinctive_framings.csv", index=False)

show_complete("Distinctiveness outputs saved")


In [ ]:
# pretty_label — imported from dialogue_utils (see Cell 5)
pass

---
# SECTION 7B: AI benefit dynamics over time
---

In [ ]:
# normalized_entropy, hhi, topk_share, parse_year — imported from dialogue_utils (see Cell 5)
pass

In [ ]:
# @title Visualise AI benefit trajectory over time

import matplotlib.pyplot as plt
import numpy as np

benefit_traj2 = benefit_traj.sort_values("year").reset_index(drop=True)

# Center trajectory on first year
x0, y0 = benefit_traj2.loc[0, ["pc1", "pc2"]]
dx = benefit_traj2["pc1"] - x0
dy = benefit_traj2["pc2"] - y0

benefit_years_arr = benefit_traj2["year"].to_numpy()

plt.figure(figsize=(6, 6))
sc = plt.scatter(dx, dy, c=benefit_years_arr, cmap="viridis", s=80)
plt.plot(dx, dy, alpha=0.6)

plt.axhline(0, color="grey", linewidth=1, alpha=0.5)
plt.axvline(0, color="grey", linewidth=1, alpha=0.5)

plt.xlabel("Displacement in benefit space (PC1)")
plt.ylabel("Displacement in benefit space (PC2)")
plt.title("AI benefit profile over time\n(displacement from initial position)")

cbar = plt.colorbar(sc)
cbar.set_label("Year")

plt.tight_layout()
plt.show()


In [ ]:
# @title Analyse AI benefit salience trajectories

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

benefit_ai_rel = benefit_ai_year.div(benefit_ai_year.sum(axis=1), axis=0).fillna(0)

# Compute linear trend (slope) for each cluster
# Note: this uses a year-index, not the actual years, so non-uniform gaps
# are treated as uniform. See methodology limitations.
t = np.arange(len(benefit_ai_rel))
slopes = {}
for c in benefit_ai_rel.columns:
    y = benefit_ai_rel[c].to_numpy()
    if np.all(y == 0):
        slopes[c] = np.nan
    else:
        slopes[c] = np.polyfit(t, y, 1)[0]

trend = pd.Series(slopes).dropna().sort_values(ascending=False)

# Select top rising and declining clusters
N = 8
top_up = trend.head(N)
top_down = trend.tail(N)

# Plot trajectories
plt.figure(figsize=(11, 5))

# Rising
plt.subplot(1, 2, 1)
for c in top_up.index:
    plt.plot(benefit_ai_rel.index, benefit_ai_rel[c], marker="o", label=c)
plt.title("AI benefits increasing in relative salience")
plt.xlabel("Year")
plt.ylabel("Share of AI attention")
plt.legend(fontsize=9)

# Declining
plt.subplot(1, 2, 2)
for c in top_down.index:
    plt.plot(benefit_ai_rel.index, benefit_ai_rel[c], marker="o", label=c)
plt.title("AI benefits decreasing in relative salience")
plt.xlabel("Year")
plt.ylabel("Share of AI attention")
plt.legend(fontsize=9)

plt.tight_layout()
plt.show()

display(
    pd.DataFrame({
        "slope": trend
    }).assign(direction=lambda d: np.where(d["slope"] > 0, "rising", "declining"))
)


In [ ]:
# @title Visualise AI benefit salience over time
# Rows = benefit clusters
# Columns = years
# Values = relative salience within AI discourse (row-normalised per year)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1) Normalise within each year (relative attention)
benefit_ai_rel = benefit_ai_year.div(benefit_ai_year.sum(axis=1), axis=0).fillna(0)

# 2) Select top N clusters by overall AI salience
N = 20
top_clusters = benefit_ai_rel.sum(axis=0).sort_values(ascending=False).head(N).index

heat = benefit_ai_rel[top_clusters]

# 3) Order clusters by when they peak
peak_year_idx = heat.values.argmax(axis=0)
order = np.argsort(peak_year_idx)
heat = heat.iloc[:, order]

heat_T = heat.T

plt.figure(figsize=(12, 7))
im = plt.imshow(
    heat_T,
    aspect="auto",
    cmap="viridis"
)

plt.colorbar(im, label="Share of AI attention (within year)")

plt.yticks(
    ticks=np.arange(len(heat_T.index)),
    labels=heat_T.index
)
plt.xticks(
    ticks=np.arange(len(heat_T.columns)),
    labels=heat_T.columns,
    rotation=45
)

plt.xlabel("Year")
plt.ylabel("Benefit cluster")
plt.title("AI public benefits over time\n(relative salience within AI discourse)")

plt.tight_layout()
plt.show()


---
# SECTION 8B: The stable core of public technology benefits
---

In [ ]:
# @title Visualise the stable core of public technology benefits

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# ---- prerequisites ----
if "benefit_cluster_entropy" not in globals():
    raise NameError("benefit_cluster_entropy not found. Run cell 59 first.")
if "benefits_df" not in globals():
    raise NameError("benefits_df not found. Run benefit extraction first.")

TECH_COL = "technology_meta"
ENTROPY_THRESHOLD = 0.5  # canonical cross-cutting threshold

# ---- build cluster-level dataframe ----
cluster_size = benefits_df["cluster_id"].value_counts()

# Note: benefit_cluster_entropy stores RAW entropy. We need normalized entropy
# for the 0.5 threshold to be meaningful. Use normalized_entropy_benefits.
df = (
    pd.DataFrame({
        "cluster_id": list(normalized_entropy_benefits.keys()),
        "entropy": [normalized_entropy_benefits[c] for c in normalized_entropy_benefits.keys()],
    })
    .set_index("cluster_id")
)

df["size"] = cluster_size
df["size"] = df["size"].fillna(0)
df = df[df["size"] > 0]

if "BENEFIT_CLUSTER_LABELS" in globals():
    df["label"] = df.index.map(lambda c: BENEFIT_CLUSTER_LABELS.get(c, f"Cluster {c}"))
else:
    df["label"] = df.index.map(lambda c: f"Cluster {c}")

entropy_vals = df["entropy"].to_numpy(dtype=float)
size = df["size"].to_numpy(dtype=float)
labels = df["label"].to_numpy(dtype=str)

# ---- define cross-cutting + stable core ----
cross_cutting = entropy_vals >= ENTROPY_THRESHOLD
size_thresh = np.quantile(size, 0.75)
stable_core = cross_cutting & (size >= size_thresh)

score = entropy_vals * np.log1p(size)
annotate_idx = np.argsort(score)[::-1][:12]

plt.figure(figsize=(10, 7))
ax = plt.gca()

core_region = Rectangle(
    (ENTROPY_THRESHOLD, size_thresh),
    1.0 - ENTROPY_THRESHOLD,
    size.max() * 1.05 - size_thresh,
    alpha=0.12,
    zorder=0
)
ax.add_patch(core_region)

ax.text(
    ENTROPY_THRESHOLD + 0.01,
    size.max() * 1.02,
    "Stable core\n(cross-cutting + high frequency)",
    fontsize=10,
    va="top"
)

plt.scatter(entropy_vals[~cross_cutting], size[~cross_cutting], alpha=0.6, s=35, label="More tech-specific clusters")
plt.scatter(entropy_vals[cross_cutting], size[cross_cutting], alpha=0.8, s=45, label="More cross-cutting clusters")
plt.scatter(entropy_vals[stable_core], size[stable_core], s=90, label="Stable core")

for i in annotate_idx:
    plt.annotate(labels[i], (entropy_vals[i], size[i]), textcoords="offset points", xytext=(6, 4), ha="left", fontsize=9)

plt.axvline(ENTROPY_THRESHOLD, linestyle="--", linewidth=1)
plt.axhline(size_thresh, linestyle="--", linewidth=1)

plt.xlabel("Technology-distribution entropy (higher = more cross-cutting)")
plt.ylabel("Cluster frequency / size (higher = more recurrent)")
plt.title("Figure: The Stable Core of Public Technology Benefits")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "benefit_stable_core_scatter.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.show()


In [ ]:
# @title Visualise the benefit embedding space

import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

ARTIFACT_DIR = globals().get("ARTIFACT_DIR", "analysis_artifacts")
EXCEL_PATH = globals().get("EXCEL_PATH", os.path.join(ARTIFACT_DIR, "public_dialogue_analysis_v9.xlsx"))

if "benefit_salience_df" not in globals():
    raise NameError("benefit_salience_df not found. Run Section 6.1a (benefits) to compute it first.")

tech_by_cluster = benefit_salience_df.copy()
tech_by_cluster = tech_by_cluster.apply(pd.to_numeric, errors="coerce").fillna(0)

# Convert to clusters x technologies
cluster_features = tech_by_cluster.T
clusters = cluster_features.index.astype(str).to_numpy()
X = cluster_features.to_numpy(dtype=float)

print("Cluster feature matrix:", X.shape, "(clusters x technologies)")

size_vals = cluster_features.sum(axis=1).to_numpy()
size_scaled = 30 + 200 * (np.sqrt(size_vals) / (np.sqrt(size_vals).max() + 1e-9))

# Load benefit framing lens mapping if available
lens_map = None
if "BENEFIT_FRAMING_LENS_MAPPINGS" in globals():
    lens_map_raw = BENEFIT_FRAMING_LENS_MAPPINGS
    lens_map = {}
    for lens_name, data in lens_map_raw.items():
        for cid in data.get('cluster_ids', []):
            lens_map[str(cid)] = str(lens_name)
else:
    json_path = os.path.join(ARTIFACT_DIR, "benefit_framing_lens_mappings.json")
    if os.path.exists(json_path):
        with open(json_path, "r") as f:
            lens_map = json.load(f)
        if lens_map and all(isinstance(v, list) for v in lens_map.values()):
            inv = {}
            for lens, clist in lens_map.items():
                for c in clist:
                    inv[str(c)] = str(lens)
            lens_map = inv
        else:
            lens_map = {str(k): str(v) for k, v in lens_map.items()}

if lens_map is None:
    lens_labels = np.array(["(no lens)"] * len(clusters), dtype=object)
else:
    lens_labels = np.array([lens_map.get(c, "(unmapped)") for c in clusters], dtype=object)

unique_lenses = pd.unique(lens_labels)
lens_to_code = {lens: i for i, lens in enumerate(unique_lenses)}
codes = np.array([lens_to_code[l] for l in lens_labels], dtype=int)

# PCA projection
X_mean = X.mean(axis=0, keepdims=True)
X_std = X.std(axis=0, keepdims=True) + 1e-9
Xz = (X - X_mean) / X_std

from sklearn.decomposition import PCA
pca = PCA(n_components=2, random_state=0)
coords = pca.fit_transform(Xz)
x, y = coords[:, 0], coords[:, 1]
expl = pca.explained_variance_ratio_

plt.figure(figsize=(10, 7))
ax = plt.gca()

cmap = plt.get_cmap("tab20", len(unique_lenses))
sc = ax.scatter(x, y, s=size_scaled, c=codes, cmap=cmap, alpha=0.85)

ax.set_xlabel(f"Benefit space (PC1; {expl[0]:.0%} var)")
ax.set_ylabel(f"Benefit space (PC2; {expl[1]:.0%} var)")
ax.set_title("Figure: Benefit Space of Public Technology Benefits\n(Clusters projected by technology-salience profile; size ~ frequency proxy)")

topN = 12
top_idx = np.argsort(size_vals)[::-1][:topN]
for i in top_idx:
    ax.annotate(clusters[i], (x[i], y[i]), textcoords="offset points", xytext=(6, 4), ha="left", fontsize=9)

lens_counts = pd.Series(lens_labels).value_counts()
top_lenses = lens_counts.index[:12].tolist() if len(lens_counts) > 12 else lens_counts.index.tolist()

legend_handles = [
    Line2D([0], [0], marker='o', linestyle='', markersize=8,
           markerfacecolor=cmap(lens_to_code[lens]), markeredgecolor='none', label=lens)
    for lens in top_lenses
]
ax.legend(handles=legend_handles, title="Framing lens (top shown)", loc="best", frameon=True)

plt.tight_layout()
out_path = "figure_benefit_space_pca.png"
plt.savefig(out_path, dpi=200)
plt.show()

print("Saved figure to:", out_path)


# PART III: Risk–benefit comparison (light-touch integration)

This section compares outputs from PART I (concerns) and PART II (benefits) without merging embedding spaces.

---


In [ ]:
# _volume_table — imported from dialogue_utils (see Cell 5)
pass

In [ ]:
# _top_clusters — imported from dialogue_utils (see Cell 5)
pass

If you want per-technology views, filter `concerns_df` / `benefits_df` by `technology` and rerun the top-cluster cell.


# PART IV: Robustness & sensitivity (concerns and benefits)

These checks validate clustering and headline outputs for both evaluative spaces.

---



## IV.A Robustness & sensitivity — concerns


# SECTION 9: Robustness checks and validation

In [ ]:
# clusters_to_labels, clusters_to_lenses, html_escape — imported from dialogue_utils (see Cell 5)
pass

In [ ]:
# @title (v16) Export human validation sample (concerns)
# Issue 12 from the audit: the paper makes methodological claims about LLM-
# mediated qualitative scaling but does not include a human inter-rater check.
# The traceability output above gives every paragraph + extracted concerns +
# cluster + lens. This cell exports a stratified-by-technology random sample
# of paragraphs ready for hand-coding.
#
# To use: download the Excel file, fill the human columns, re-import, and
# compute agreement statistics (Cohen's kappa or simple agreement) against
# the LLM extractions.

show_status("Exporting human validation sample (concerns)...")

VALIDATION_SAMPLE_N = 250

# Stratified sample by technology, with a floor per technology so small-N
# technologies are not invisible.
per_tech_floor = 10
sample_pieces = []
for tech, group in chunks_df.groupby("technology_meta"):
    n_for_tech = max(per_tech_floor, int(round(VALIDATION_SAMPLE_N * len(group) / len(chunks_df))))
    n_for_tech = min(n_for_tech, len(group))
    sample_pieces.append(group.sample(n=n_for_tech, random_state=RANDOM_SEED))

validation_sample = pd.concat(sample_pieces, ignore_index=True)
# Trim if oversized after the floor
if len(validation_sample) > VALIDATION_SAMPLE_N * 1.5:
    validation_sample = validation_sample.sample(int(VALIDATION_SAMPLE_N * 1.5), random_state=RANDOM_SEED)

# Attach LLM extractions for each sampled chunk
sample_concerns = (concerns_df.groupby("chunk_id")["concern"]
                   .apply(lambda s: " | ".join(s))
                   .rename("llm_concerns"))
validation_sample = validation_sample.merge(sample_concerns, on="chunk_id", how="left")
validation_sample["llm_concerns"] = validation_sample["llm_concerns"].fillna("(NO_CONCERN)")
validation_sample["llm_concern_count"] = validation_sample["llm_concerns"].apply(
    lambda s: 0 if s == "(NO_CONCERN)" else s.count("|") + 1
)

# Empty columns for the human coder
for col in ["human_any_concern", "human_concerns", "human_retain_llm_extraction", "human_notes"]:
    validation_sample[col] = ""

cols_for_export = [
    "chunk_id", "source_file", "technology_meta", "year", "word_count",
    "text", "llm_concerns", "llm_concern_count",
    "human_any_concern", "human_concerns", "human_retain_llm_extraction", "human_notes",
]
validation_sample = validation_sample[cols_for_export]

# Strip control characters that openpyxl refuses to write.
# These are PDF-extraction artefacts (vertical tabs, form feeds, etc.) and
# carry no semantic content. We clean a copy so the in-memory dataframe is
# untouched.
import re
ILLEGAL_XLSX_CHARS = re.compile(r"[\x00-\x08\x0B-\x0C\x0E-\x1F]")

def _clean_for_xlsx(v):
    if isinstance(v, str):
        return ILLEGAL_XLSX_CHARS.sub("", v)
    return v

validation_sample_xlsx = validation_sample.applymap(_clean_for_xlsx)
validation_sample_xlsx.to_excel(OUTPUT_FOLDER / "human_validation_sample_concerns.xlsx", index=False)
validation_sample.to_csv(OUTPUT_FOLDER / "human_validation_sample_concerns.csv", index=False)

print(f"Exported {len(validation_sample)} chunks for human validation.")
print(f"Stratified by technology; per-technology floor = {per_tech_floor}.")
print()
print("Suggested human columns:")
print("  human_any_concern: yes / no / unclear")
print("  human_concerns: pipe-separated phrases the human extracted")
print("  human_retain_llm_extraction: yes / no / partial")
print("  human_notes: free text")

show_complete("Human validation sample exported. See human_validation_sample_concerns.xlsx")


# Paper assets

The cells below regenerate the paper-ready tables and figures from the analysis outputs above. Each cell writes its output to the `paper_assets/` subfolder of `OUTPUT_FOLDER`.

These cells are deliberately self-contained: they re-load the relevant CSVs from `OUTPUT_FOLDER` rather than relying on in-memory variables. This means they can be re-run individually without re-running the full pipeline, and they remain valid if you re-run the analysis with different parameters.

The four assets produced here are:

- **Table 1**: top 20 cross-cutting concern clusters (the "shared grammar" finding).
- **Table 2**: AI distinctiveness, two panels — cluster-level (top 10 over- and under-indexed) and lens-level (both baselines).
- **Figure 3**: AI vs non-AI distinctive clusters, paired horizontal bars.
- **Figure 4**: AI concern profile across four time windows, heatmap.

Figures 1 and 2 are produced earlier in the notebook (the stable-core scatter cells now save PNG copies; the lens radar cells now save PNG copies alongside their HTML versions).


In [ ]:
# @title (paper) Table 1: Top 20 cross-cutting concern clusters
# Joins shared_concerns_top20.csv with cluster_summary.csv to get labels
# alongside the shared-concern statistics.

PAPER_ASSETS = OUTPUT_FOLDER / "paper_assets"
PAPER_ASSETS.mkdir(exist_ok=True)

_sc = pd.read_csv(OUTPUT_FOLDER / "shared_concerns_top20.csv")
_cs = pd.read_csv(OUTPUT_FOLDER / "cluster_summary.csv")

_t1 = _sc.merge(_cs[["cluster_id", "label", "size"]], on="cluster_id", how="left")
_t1 = _t1[["cluster_id", "label", "size", "global_prevalence", "tech_entropy"]]
_t1 = _t1.rename(columns={
    "cluster_id": "Cluster ID",
    "label": "Cluster label",
    "size": "N phrases",
    "global_prevalence": "Share of all concern phrases",
    "tech_entropy": "Technology entropy (raw)",
})
_t1["Share of all concern phrases"] = (_t1["Share of all concern phrases"] * 100).round(2).astype(str) + "%"
_t1["Technology entropy (raw)"] = _t1["Technology entropy (raw)"].round(3)

print("=== Table 1: Top 20 cross-cutting concern clusters ===\n")
print(_t1.to_string(index=False))
_total_share = _sc["global_prevalence"].sum() * 100
print(f"\nTop 20 share = {_total_share:.1f}% of all concern phrases.")

_t1.to_csv(PAPER_ASSETS / "table1_top_cross_cutting_concerns.csv", index=False)
_t1.to_excel(PAPER_ASSETS / "table1_top_cross_cutting_concerns.xlsx", index=False)
print(f"Saved: {PAPER_ASSETS / 'table1_top_cross_cutting_concerns.xlsx'}")


In [ ]:
# @title (paper) Table 2: AI distinctiveness (clusters and lenses)
# Two-panel table. Panel A: cluster-level top 10 over-indexed and top 10
# under-indexed. Panel B: lens-level distinctiveness under both baselines.

PAPER_ASSETS = OUTPUT_FOLDER / "paper_assets"
PAPER_ASSETS.mkdir(exist_ok=True)

_clusters = pd.read_csv(OUTPUT_FOLDER / "ai_distinctive_concerns.csv")
_lenses_tech = pd.read_csv(OUTPUT_FOLDER / "ai_distinctive_framings.csv")
_lenses_doc = pd.read_csv(OUTPUT_FOLDER / "ai_vs_nonai_lens_doc_weighted.csv")

# Panel A
_A = _clusters.sort_values("ai_minus_nonai", ascending=False).reset_index(drop=True)
_A["AI share (%)"] = (_A["ai_share"] * 100).round(1)
_A["Non-AI share (%)"] = (_A["nonai_share"] * 100).round(1)
_A["Difference (pp)"] = (_A["ai_minus_nonai"] * 100).round(1)
_A["Direction"] = _A["Difference (pp)"].apply(lambda x: "AI over-indexed" if x > 0 else "AI under-indexed")
_panel_a = _A[["cluster_id", "cluster_label", "AI share (%)", "Non-AI share (%)", "Difference (pp)", "Direction"]]
_panel_a = _panel_a.rename(columns={"cluster_id": "Cluster ID", "cluster_label": "Cluster label"})

# Panel B
_B = _lenses_tech.merge(
    _lenses_doc[["lens", "non_ai_doc_weighted_share", "ai_minus_doc_weighted"]],
    left_on="framing_lens", right_on="lens", how="left"
).drop(columns=["lens"])
_B = _B.sort_values("ai_minus_nonai", ascending=False).reset_index(drop=True)
_B["AI share (%)"] = (_B["ai_share"] * 100).round(1)
_B["Non-AI share, tech-weighted (%)"] = (_B["nonai_share"] * 100).round(1)
_B["Non-AI share, doc-weighted (%)"] = (_B["non_ai_doc_weighted_share"] * 100).round(1)
_B["Difference, tech-weighted (pp)"] = (_B["ai_minus_nonai"] * 100).round(1)
_B["Difference, doc-weighted (pp)"] = (_B["ai_minus_doc_weighted"] * 100).round(1)
_panel_b = _B[["framing_lens", "AI share (%)",
               "Non-AI share, tech-weighted (%)", "Difference, tech-weighted (pp)",
               "Non-AI share, doc-weighted (%)", "Difference, doc-weighted (pp)"]]
_panel_b = _panel_b.rename(columns={"framing_lens": "Framing lens"})

print("=== Table 2 Panel A: Cluster-level AI distinctiveness ===")
print(_panel_a.to_string(index=False))
print("\n=== Table 2 Panel B: Lens-level AI distinctiveness (both baselines) ===")
print(_panel_b.to_string(index=False))

# Save: CSV with both panels stacked
import io as _io
_csv_buf = _io.StringIO()
_csv_buf.write("Panel A: Cluster-level AI distinctiveness\n")
_panel_a.to_csv(_csv_buf, index=False)
_csv_buf.write("\n\nPanel B: Lens-level AI distinctiveness (both baselines)\n")
_panel_b.to_csv(_csv_buf, index=False)
(PAPER_ASSETS / "table2_ai_distinctiveness.csv").write_text(_csv_buf.getvalue())

# Save: Excel with two sheets
with pd.ExcelWriter(PAPER_ASSETS / "table2_ai_distinctiveness.xlsx") as _w:
    _panel_a.to_excel(_w, sheet_name="Panel A - Clusters", index=False)
    _panel_b.to_excel(_w, sheet_name="Panel B - Lenses", index=False)

print(f"\nSaved: {PAPER_ASSETS / 'table2_ai_distinctiveness.xlsx'}")
print(f"       {PAPER_ASSETS / 'table2_ai_distinctiveness.csv'}")


In [ ]:
# @title (paper) Figure 3: AI vs non-AI distinctive clusters
# Paired horizontal bars showing the top 20 AI-distinctive concern clusters
# (10 most over-indexed, 10 most under-indexed) with AI share and non-AI
# share visible side by side.

import numpy as _np
import matplotlib.pyplot as _plt

PAPER_ASSETS = OUTPUT_FOLDER / "paper_assets"
PAPER_ASSETS.mkdir(exist_ok=True)

_dc = pd.read_csv(OUTPUT_FOLDER / "ai_distinctive_concerns.csv")
_dc = _dc.sort_values("ai_minus_nonai", ascending=True).reset_index(drop=True)
_dc["ai_pct"] = _dc["ai_share"] * 100
_dc["nonai_pct"] = _dc["nonai_share"] * 100
_dc["diff_pp"] = _dc["ai_minus_nonai"] * 100

_n = len(_dc)
_labels = _dc["cluster_label"].tolist()

fig, ax = _plt.subplots(figsize=(10.5, max(7, _n * 0.42)))
_bar_h = 0.36
_y = _np.arange(_n)
_ai_color = "#1f77b4"
_nonai_color = "#8a8a8a"

ax.barh(_y - _bar_h/2, _dc["ai_pct"], height=_bar_h, color=_ai_color,
        label="AI", edgecolor="white", linewidth=0.5)
ax.barh(_y + _bar_h/2, _dc["nonai_pct"], height=_bar_h, color=_nonai_color,
        label="Non-AI (technology-weighted)", edgecolor="white", linewidth=0.5)

ax.set_yticks(_y)
ax.set_yticklabels(_labels, fontsize=9.5)
ax.set_xlabel("Share of concern phrases (%)", fontsize=10)
ax.tick_params(axis="x", labelsize=9)

_xmax = max(_dc["ai_pct"].max(), _dc["nonai_pct"].max()) * 1.18
ax.set_xlim(0, _xmax)

for i, row in _dc.iterrows():
    diff = row["diff_pp"]
    sign = "+" if diff > 0 else ""
    color_diff = _ai_color if diff > 0 else "#666666"
    weight = "bold" if abs(diff) > 1.5 else "normal"
    longest = max(row["ai_pct"], row["nonai_pct"])
    ax.annotate(f"Δ {sign}{diff:.1f}pp", xy=(longest + 0.15, i),
                ha="left", va="center", fontsize=9, color=color_diff, fontweight=weight)

_n_under = (_dc["diff_pp"] < 0).sum()
ax.axhline(_n_under - 0.5, color="#bbbbbb", linewidth=0.7, linestyle="--", zorder=0)

ax.text(_xmax * 0.97, _n_under + (_n - _n_under) / 2 - 0.5, "AI over-indexed",
        ha="right", va="center", fontsize=10, color=_ai_color,
        fontweight="bold", alpha=0.55, rotation=90)
ax.text(_xmax * 0.97, _n_under / 2 - 0.5, "AI under-indexed",
        ha="right", va="center", fontsize=10, color="#666666",
        fontweight="bold", alpha=0.55, rotation=90)

ax.set_title("AI vs non-AI concern profile: 20 most distinctive clusters\n"
             "Top 10 most AI-over-indexed and top 10 most AI-under-indexed",
             fontsize=11, pad=10)
ax.legend(loc="lower right", fontsize=9, frameon=True, framealpha=0.95)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#dddddd")
ax.spines["bottom"].set_color("#cccccc")
ax.grid(axis="x", linestyle=":", linewidth=0.5, color="#dddddd", alpha=0.7, zorder=0)
ax.set_axisbelow(True)

fig.text(0.02, 0.005,
         "Δ = AI share minus non-AI share (percentage points). "
         "Non-AI baseline is technology-weighted: each non-AI technology contributes equally.",
         fontsize=8, color="#555555", ha="left")

_plt.tight_layout(rect=[0, 0.03, 1, 1])
_plt.savefig(PAPER_ASSETS / "figure3_ai_distinctive_clusters.png",
             dpi=200, bbox_inches="tight", facecolor="white")
_plt.show()
print(f"Saved: {PAPER_ASSETS / 'figure3_ai_distinctive_clusters.png'}")


In [ ]:
# @title (paper) Figure 4: AI concern profile across four time windows
# Heatmap of cluster shares within each time window for the union of clusters
# that appeared in the top 10 of any window. Rows are ordered by which
# window each cluster reaches its peak in, so the diagonal pattern reads
# as the "content shifts within stable spread" finding.

import json as _json
import numpy as _np
import matplotlib.pyplot as _plt
import matplotlib.colors as _mcolors

PAPER_ASSETS = OUTPUT_FOLDER / "paper_assets"
PAPER_ASSETS.mkdir(exist_ok=True)

# Load traceability and the existing top-clusters-per-window file
_trace = pd.read_csv(OUTPUT_FOLDER / "concern_traceability_paragraphs.csv")
_top_window = pd.read_csv(OUTPUT_FOLDER / "ai_concern_window_top_clusters.csv")
_summary = pd.read_csv(OUTPUT_FOLDER / "cluster_summary.csv")

def _parse_listcol(s):
    if pd.isna(s) or s == "" or s == "[]":
        return []
    try:
        return _json.loads(s.replace("'", '"'))
    except Exception:
        s = s.strip("[]")
        return [x.strip().strip("'\"") for x in s.split(",")] if s else []

def _assign_window(year):
    if pd.isna(year):
        return None
    y = int(year)
    if y <= 2017: return "2004-2017"
    if y <= 2020: return "2018-2020"
    if y <= 2023: return "2021-2023"
    return "2024-2025"

_ai = _trace[_trace["technology_meta"] == "AI"].copy()
_ai["cluster_ids_parsed"] = _ai["cluster_ids"].apply(_parse_listcol)
_ai["window"] = _ai["year_meta"].apply(_assign_window)

_exploded = _ai[["window", "cluster_ids_parsed"]].explode("cluster_ids_parsed").dropna()
_exploded["cluster_id"] = pd.to_numeric(_exploded["cluster_ids_parsed"], errors="coerce")
_exploded = _exploded.dropna(subset=["cluster_id"])
_exploded["cluster_id"] = _exploded["cluster_id"].astype(int)

_counts = _exploded.groupby(["window", "cluster_id"]).size().reset_index(name="n")
_window_totals = _exploded.groupby("window").size().reset_index(name="window_total")
_counts = _counts.merge(_window_totals, on="window")
_counts["share_within_window"] = _counts["n"] / _counts["window_total"]

_union = _top_window["cluster_id"].unique()
_hm = _counts[_counts["cluster_id"].isin(_union)].copy()

_window_order = ["2004-2017", "2018-2020", "2021-2023", "2024-2025"]
_pivot = _hm.pivot(index="cluster_id", columns="window", values="share_within_window").fillna(0)
_pivot = _pivot[_window_order]

_labels_map = dict(zip(_summary["cluster_id"], _summary["label"]))
_pivot["label"] = _pivot.index.map(_labels_map)

def _peak_window(row):
    shares = row[_window_order]
    return _window_order.index(shares.idxmax())

_pivot["peak_window"] = _pivot.apply(_peak_window, axis=1)
_pivot["peak_share"] = _pivot[_window_order].max(axis=1)
_pivot = _pivot.sort_values(["peak_window", "peak_share"], ascending=[True, False])

# Save the underlying data
_pivot[_window_order + ["label"]].to_csv(PAPER_ASSETS / "figure4_data.csv")

# Render
_data = _pivot[_window_order].values * 100
_labels = _pivot["label"].tolist()
_n = len(_labels)

fig, ax = _plt.subplots(figsize=(8.5, max(7, _n * 0.35)))
_vmax = _np.percentile(_data[_data > 0], 95)
_im = ax.imshow(_data, aspect="auto", cmap=_plt.get_cmap("Blues"),
                norm=_mcolors.Normalize(vmin=0, vmax=_vmax))

ax.set_xticks(range(len(_window_order)))
ax.set_xticklabels(_window_order, fontsize=10)
ax.set_yticks(range(_n))
ax.set_yticklabels(_labels, fontsize=9)
ax.tick_params(axis="x", top=True, bottom=False, labeltop=True, labelbottom=False)

for i in range(_n):
    for j in range(len(_window_order)):
        v = _data[i, j]
        if v == 0:
            ax.text(j, i, "·", ha="center", va="center", fontsize=8.5, color="#bbbbbb")
        else:
            color = "white" if (v / _vmax) > 0.55 else "#222222"
            ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=8.5, color=color)
        if v > _vmax:
            ax.text(j + 0.35, i - 0.35, "▲", ha="center", va="center", fontsize=7, color="white")

for i in range(_n + 1):
    ax.axhline(i - 0.5, color="white", linewidth=0.6)
for j in range(len(_window_order) + 1):
    ax.axvline(j - 0.5, color="white", linewidth=0.6)

cbar = _plt.colorbar(_im, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label("Share of AI concern phrases in window (%)", fontsize=9)
cbar.ax.tick_params(labelsize=8)

ax.set_title(f"AI concern profile across four time windows\n"
             f"(union of top-10 clusters from each window; n={_n} clusters)",
             fontsize=11, pad=12)

fig.text(0.02, 0.01,
         "Rows ordered by the time window in which each cluster reaches its peak share. "
         "Cells show share of AI concern phrases within window. · = no phrases. "
         f"▲ = above colour-scale cap of {_vmax:.1f}%.",
         fontsize=7.5, color="#555555", ha="left")

_plt.tight_layout(rect=[0, 0.04, 1, 1])
_plt.savefig(PAPER_ASSETS / "figure4_ai_concerns_across_windows.png",
             dpi=200, bbox_inches="tight", facecolor="white")
_plt.show()
print(f"Saved: {PAPER_ASSETS / 'figure4_ai_concerns_across_windows.png'}")


# Supplementary analyses (appendix)

The cells below produce supplementary robustness checks, sensitivity analyses,
and detailed diagnostics referenced in the paper appendix and supplementary
materials. They are **not** required for the main paper figures or tables.

This section covers:
- Robustness of the stable core and AI fingerprint findings
- Volatility of AI concern/benefit rankings under perturbation
- Lexical and category novelty over time
- Sensitivity across alternative cluster counts (k=60, 75, 90)

Run these if you want full appendix outputs; otherwise the main analysis
above is self-contained.

In [ ]:
# normalized_entropy, ai_fingerprint_over_crosscut — imported from dialogue_utils (see Cell 5)
pass

In [ ]:
# @title Assess volatility of AI concern rankings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


ai_rel = ai_year.div(ai_year.sum(axis=1), axis=0).fillna(0)

ranks = ai_rel.rank(axis=1, ascending=False, method="average")

rank_diff = ranks.diff().abs()

volatility = rank_diff.mean(axis=1)

vol_df = pd.DataFrame({
    "year": volatility.index,
    "mean_rank_change": volatility.values
}).dropna()

plt.figure(figsize=(9, 4.5))
plt.plot(vol_df["year"], vol_df["mean_rank_change"], marker="o")

plt.xlabel("Year")
plt.ylabel("Mean absolute rank change")
plt.title("Stability of AI public concerns over time\n(lower values = greater normalisation)")
plt.tight_layout()
plt.show()

display(vol_df)


In [ ]:
# parse_year, tokenize — imported from dialogue_utils (see Cell 5)
pass

In [ ]:
# @title Assess novelty of concern types in AI dialogues

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ai_year exists: years × clusters (raw salience)
present = (ai_year > 0).astype(int)

seen = set()
rows = []
for y in present.index:
    active = set(present.columns[present.loc[y] > 0])
    new = active - seen
    rows.append({
        "year": int(y),
        "active_clusters": len(active),
        "new_clusters": len(new),
        "new_cluster_share": len(new) / max(len(active), 1)
    })
    seen |= active

nov2 = pd.DataFrame(rows)

plt.figure(figsize=(9, 4.5))
plt.plot(nov2["year"], nov2["new_cluster_share"], marker="o")
plt.xlabel("Year")
plt.ylabel("Share of newly appearing concern clusters")
plt.title("Novelty of concern-types in AI discourse over time")
plt.tight_layout()
plt.show()

display(nov2)


---
# SECTION 10: Sensitivity analysis: number of clusters
We re-run *selected headline outputs* for **k = 60** and **k = 90** (baseline **k = 75**) to assess whether substantive conclusions depend on clustering resolution.

**Notes:**
- Cluster IDs are not expected to match across k.
- For framing-lens comparisons, we map new clusters to the closest baseline (k=75) cluster centroid (cosine similarity) and inherit baseline lens memberships. This provides a consistent lens vocabulary without re-running LLM labelling.


In [ ]:
# @title Compare results across alternative cluster counts (60, 75, 90)

show_status("Running sensitivity analysis for cluster counts...")

from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import entropy as shannon_entropy

TECH_COL = 'technology_meta'
ks = [60, 75, 90]

# --- Ensure baseline_centroids exist (k=75 baseline) ---
BASELINE_K = 75

if "baseline_centroids" not in globals():
    km_base = KMeans(n_clusters=BASELINE_K, random_state=RANDOM_SEED, n_init=10)
    _ = km_base.fit_predict(embeddings_normalized)
    baseline_centroids = km_base.cluster_centers_.copy()

# Baseline lens membership: baseline cluster -> set(lenses)
baseline_cluster_to_lenses = {}
for lens_name, data in FRAMING_LENS_MAPPINGS.items():
    for cid in data['cluster_ids']:
        baseline_cluster_to_lenses.setdefault(cid, set()).add(lens_name)

def run_for_k(k: int):
    # Re-cluster on the same (normalised) embeddings
    km = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10)
    labels = km.fit_predict(embeddings_normalized)
    centroids = km.cluster_centers_.copy()

    df = concerns_df.copy()
    df['cluster_id_k'] = labels

    # --- Stable core analogue: entropy across technologies × global prevalence
    global_prev = df['cluster_id_k'].value_counts(normalize=True)
    ent = {}
    for cid in range(k):
        mask = df['cluster_id_k'] == cid
        tech_counts = df.loc[mask, TECH_COL].value_counts()
        probs = (tech_counts / tech_counts.sum()).values if tech_counts.sum()>0 else np.array([])
        ent[cid] = float(shannon_entropy(probs)) if len(probs) > 1 else 0.0

    stable_df = pd.DataFrame({
        'cluster_id_k': list(range(k)),
        'tech_entropy': [ent[i] for i in range(k)],
        'global_prevalence': [float(global_prev.get(i,0)) for i in range(k)]
    }).sort_values(['global_prevalence','tech_entropy'], ascending=False)

    # Plot
    plt.figure(figsize=(7,5))
    plt.scatter(stable_df['tech_entropy'], stable_df['global_prevalence'])
    plt.xlabel("Entropy across technologies")
    plt.ylabel("Share of all extracted concerns")
    plt.title(f"Stable core structure (k={k})")
    plt.tight_layout()
    plt.savefig(OUTPUT_FOLDER / f"sensitivity_stable_core_k{k}.png")
    plt.show()

    stable_df.to_csv(OUTPUT_FOLDER / f"sensitivity_stable_core_k{k}.csv", index=False)

    # --- AI fingerprint vs non-AI baseline on clusters (technology-weighted)
    techs = sorted(df[TECH_COL].dropna().unique().tolist())
    non_ai = [t for t in techs if t != 'AI']

    # Technology profiles
    prof = {}
    for t in techs:
        m = df[TECH_COL]==t
        total = m.sum()
        if total==0:
            continue
        counts = df.loc[m,'cluster_id_k'].value_counts()
        prof[t] = (counts/total).to_dict()
    prof_df = pd.DataFrame(prof).T.fillna(0)
    non_ai_avg = prof_df.loc[non_ai].mean(axis=0) if len(non_ai) else pd.Series(dtype=float)

    if 'AI' in prof_df.index and len(non_ai_avg)>0:
        diff = (prof_df.loc['AI'] - non_ai_avg).sort_values()
        top_low = diff.head(5).index.tolist()
        top_high = diff.tail(7).index.tolist()
        sel = top_low + top_high

        categories = [f"Cluster {c}" for c in sel]
        ai_vals = prof_df.loc['AI', sel].tolist()
        base_vals = non_ai_avg[sel].tolist()

        categories_loop = categories + [categories[0]]
        ai_loop = ai_vals + [ai_vals[0]]
        base_loop = base_vals + [base_vals[0]]

        fig = go.Figure()
        fig.add_trace(go.Scatterpolar(r=ai_loop, theta=categories_loop, fill='toself', name='AI'))
        fig.add_trace(go.Scatterpolar(r=base_loop, theta=categories_loop, fill='toself', name='Non‑AI avg (tech‑weighted)'))
        fig.update_layout(
            polar=dict(radialaxis=dict(visible=True, range=[0, max(ai_loop+base_loop)*1.1])),
            title=f"AI fingerprint vs non‑AI baseline (k={k})"
        )
        fig.write_html(OUTPUT_FOLDER / f"sensitivity_ai_fingerprint_clusters_k{k}.html")
        fig.show()

    # --- AI over time (headline): entropy / concentration of AI concern distribution by year
    ai_df = df[df[TECH_COL]=='AI'].dropna(subset=['year'])
    if len(ai_df)>0:
        yearly = []
        for yr, g in ai_df.groupby('year'):
            counts = g['cluster_id_k'].value_counts(normalize=True)
            ent_y = float(shannon_entropy(counts.values)) if len(counts)>1 else 0.0
            top10 = float(counts.nlargest(min(10,len(counts))).sum())
            hhi = float((counts.values**2).sum())
            yearly.append({'year': int(yr), 'entropy': ent_y, 'top10_share': top10, 'hhi': hhi})
        yd = pd.DataFrame(yearly).sort_values('year')
        yd.to_csv(OUTPUT_FOLDER / f"sensitivity_ai_time_metrics_k{k}.csv", index=False)

        plt.figure(figsize=(8,4))
        plt.plot(yd['year'], yd['entropy'], marker='o')
        plt.title(f"AI discourse diversity over time (entropy, k={k})")
        plt.xlabel("Year")
        plt.ylabel("Entropy of cluster distribution")
        plt.tight_layout()
        plt.savefig(OUTPUT_FOLDER / f"sensitivity_ai_entropy_over_time_k{k}.png")
        plt.show()

    # --- AI vs non-AI by framing lens (mapped to baseline lens vocabulary)
    # Map new clusters to nearest baseline centroid, then inherit baseline lens memberships
    sim = cosine_similarity(centroids, baseline_centroids)
    nearest = sim.argmax(axis=1)  # baseline cluster id for each new cluster
    new_cluster_to_lenses = {}
    for new_cid in range(k):
        base_cid = int(nearest[new_cid])
        new_cluster_to_lenses[new_cid] = baseline_cluster_to_lenses.get(base_cid, set())

    # Lens salience per tech
    lens_names = list(FRAMING_LENS_MAPPINGS.keys())
    lens_by_tech = {}
    for t in techs:
        m = df[TECH_COL]==t
        total = m.sum()
        if total==0:
            continue
        lens_by_tech[t] = {ln:0.0 for ln in lens_names}
        # count concerns by lens via cluster mapping
        for cid, cnt in df.loc[m,'cluster_id_k'].value_counts().items():
            for ln in new_cluster_to_lenses.get(int(cid), set()):
                lens_by_tech[t][ln] += cnt
        # normalise
        for ln in lens_names:
            lens_by_tech[t][ln] = lens_by_tech[t][ln] / total

    lens_df = pd.DataFrame(lens_by_tech).T.fillna(0)
    lens_df.to_csv(OUTPUT_FOLDER / f"sensitivity_lens_salience_by_tech_k{k}.csv")

    if 'AI' in lens_df.index and len(non_ai)>0:
        non_ai_avg_lens = lens_df.loc[non_ai].mean(axis=0)
        cats = list(non_ai_avg_lens.index)
        ai_vals = lens_df.loc['AI', cats].tolist()
        base_vals = non_ai_avg_lens[cats].tolist()

        cats_loop = cats + [cats[0]]
        ai_loop = ai_vals + [ai_vals[0]]
        base_loop = base_vals + [base_vals[0]]

        fig = go.Figure()
        fig.add_trace(go.Scatterpolar(r=ai_loop, theta=cats_loop, fill='toself', name='AI'))
        fig.add_trace(go.Scatterpolar(r=base_loop, theta=cats_loop, fill='toself', name='Non‑AI avg (tech‑weighted)'))
        fig.update_layout(
            polar=dict(radialaxis=dict(visible=True, range=[0, max(ai_loop+base_loop)*1.1])),
            title=f"AI framing profile vs non‑AI baseline (mapped lenses, k={k})"
        )
        fig.write_html(OUTPUT_FOLDER / f"sensitivity_ai_framing_radar_k{k}.html")
        fig.show()

for k in ks:
    print(f"\n=== Sensitivity run: k={k} ===")
    run_for_k(k)

show_complete("Sensitivity runs complete (see outputs folder)")



## IV.B Robustness & sensitivity — benefits


# SECTION 9B: Robustness checks and validation (benefits)

In [ ]:
# clusters_to_labels, clusters_to_lenses, html_escape — imported from dialogue_utils (see Cell 5)
pass

In [ ]:
# _clean_for_xlsx is defined locally in the export cell above (Cell 94).
# It strips openpyxl-illegal control characters (PDF extraction artefacts)
# from Excel export values and is NOT imported from dialogue_utils.

In [ ]:
# normalized_entropy, ai_fingerprint_over_crosscut — imported from dialogue_utils (see Cell 5)
pass

In [ ]:
# @title Assess volatility of AI benefit rankings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if "benefit_ai_year" not in globals():
    raise NameError("benefit_ai_year not found. Run cell 80 first.")

ai_rel = benefit_ai_year.div(benefit_ai_year.sum(axis=1), axis=0).fillna(0)

ranks = ai_rel.rank(axis=1, ascending=False, method="average")

rank_diff = ranks.diff().abs()

volatility = rank_diff.mean(axis=1)

vol_df = pd.DataFrame({
    "year": volatility.index,
    "mean_rank_change": volatility.values
}).dropna()

plt.figure(figsize=(9, 4.5))
plt.plot(vol_df["year"], vol_df["mean_rank_change"], marker="o")

plt.xlabel("Year")
plt.ylabel("Mean absolute rank change")
plt.title("Stability of AI public benefits over time\n(lower values = greater normalisation)")
plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "benefit_ai_volatility_over_time.png", dpi=150, bbox_inches='tight')
plt.show()

display(vol_df)


In [ ]:
# parse_year, tokenize — imported from dialogue_utils (see Cell 5)
pass

In [ ]:
# @title Assess novelty of benefit types in AI dialogues

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

present = (benefit_ai_year > 0).astype(int)

seen = set()
rows = []
for y in present.index:
    active = set(present.columns[present.loc[y] > 0])
    new = active - seen
    rows.append({
        "year": int(y),
        "active_clusters": len(active),
        "new_clusters": len(new),
        "new_cluster_share": len(new) / max(len(active), 1)
    })
    seen |= active

nov2 = pd.DataFrame(rows)

plt.figure(figsize=(9, 4.5))
plt.plot(nov2["year"], nov2["new_cluster_share"], marker="o")
plt.xlabel("Year")
plt.ylabel("Share of newly appearing benefit clusters")
plt.title("Novelty of benefit-types in AI discourse over time")
plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "benefit_ai_cluster_novelty.png", dpi=150, bbox_inches='tight')
plt.show()

display(nov2)


---
# SECTION 10B: Sensitivity analysis: number of clusters (benefits)
We re-run *selected headline outputs* for **k = 60** and **k = 90** (baseline **k = 75**) to assess whether substantive conclusions depend on clustering resolution.

**Notes:**
- Cluster IDs are not expected to match across k.
- For framing-lens comparisons, we map new clusters to the closest baseline (k=75) cluster centroid (cosine similarity) and inherit baseline lens memberships. This provides a consistent lens vocabulary without re-running LLM labelling.


In [ ]:
# @title Compare benefit results across alternative cluster counts (60, 75, 90)

show_status("Running sensitivity analysis for benefit cluster counts...")

from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import entropy as shannon_entropy

TECH_COL = 'technology_meta'
ks = [60, 75, 90]

# --- Ensure baseline_centroids exist (k=75 baseline) ---
BASELINE_K = 75

if "baseline_benefit_centroids" not in globals():
    km_base = KMeans(n_clusters=BASELINE_K, random_state=RANDOM_SEED, n_init=10)
    _ = km_base.fit_predict(benefit_embeddings_normalized)
    baseline_benefit_centroids = km_base.cluster_centers_.copy()

baseline_cluster_to_lenses = {}
for lens_name, data in BENEFIT_FRAMING_LENS_MAPPINGS.items():
    for cid in data['cluster_ids']:
        baseline_cluster_to_lenses.setdefault(cid, set()).add(lens_name)

def run_for_k(k: int):
    km = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10)
    labels = km.fit_predict(benefit_embeddings_normalized)
    centroids = km.cluster_centers_.copy()

    df = benefits_df.copy()
    df['cluster_id_k'] = labels

    # Stable core analogue: entropy across technologies x global prevalence
    global_prev = df['cluster_id_k'].value_counts(normalize=True)
    ent = {}
    for cid in range(k):
        mask = df['cluster_id_k'] == cid
        tech_counts = df.loc[mask, TECH_COL].value_counts()
        probs = (tech_counts / tech_counts.sum()).values if tech_counts.sum() > 0 else np.array([])
        ent[cid] = float(shannon_entropy(probs)) if len(probs) > 1 else 0.0

    # Normalize entropy to [0, 1]
    n_techs = df[TECH_COL].nunique()
    max_ent = np.log(n_techs) if n_techs > 1 else 1.0
    norm_ent = {cid: e / max_ent for cid, e in ent.items()}

    cross_cutting = [cid for cid, e in norm_ent.items() if e >= 0.5]

    # Map this k's clusters to baseline (k=75) cluster centroids by cosine
    sims = cosine_similarity(centroids, baseline_benefit_centroids)
    best_baseline = sims.argmax(axis=1)  # for each new cluster, the closest baseline cluster

    # Inherit baseline lens membership
    cluster_to_lenses_k = {
        cid: baseline_cluster_to_lenses.get(int(best_baseline[cid]), set())
        for cid in range(k)
    }

    # Lens prevalence in this k
    lens_prev = {}
    for cid in range(k):
        mask = df['cluster_id_k'] == cid
        n = mask.sum()
        for lens in cluster_to_lenses_k[cid]:
            lens_prev[lens] = lens_prev.get(lens, 0) + n

    total = sum(lens_prev.values())
    if total > 0:
        lens_prev = {l: v / total for l, v in lens_prev.items()}

    return {
        'k': k,
        'n_cross_cutting': len(cross_cutting),
        'pct_cross_cutting': len(cross_cutting) / k,
        'lens_prev': lens_prev,
        'top_lenses': sorted(lens_prev.items(), key=lambda x: -x[1])[:5],
    }

results = {}
for k in ks:
    print(f"\n--- k = {k} ---")
    results[k] = run_for_k(k)
    r = results[k]
    print(f"  Cross-cutting clusters: {r['n_cross_cutting']} ({r['pct_cross_cutting']:.0%})")
    print("  Top lenses:")
    for lens, prev in r['top_lenses']:
        print(f"    {lens}: {prev:.1%}")

# Build comparison table
all_lenses = set()
for r in results.values():
    all_lenses.update(r['lens_prev'].keys())

comp_rows = []
for lens in sorted(all_lenses):
    row = {'framing_lens': lens}
    for k in ks:
        row[f'k={k}'] = results[k]['lens_prev'].get(lens, 0)
    comp_rows.append(row)

comp_df = pd.DataFrame(comp_rows).sort_values(f'k={BASELINE_K}', ascending=False)

print("\n--- Lens prevalence by cluster count ---")
display(comp_df)

comp_df.to_csv(OUTPUT_FOLDER / "benefit_sensitivity_lens_prevalence.csv", index=False)

summary_rows = [{
    'k': k,
    'n_cross_cutting': results[k]['n_cross_cutting'],
    'pct_cross_cutting': results[k]['pct_cross_cutting'],
} for k in ks]
pd.DataFrame(summary_rows).to_csv(OUTPUT_FOLDER / "benefit_sensitivity_summary.csv", index=False)

show_complete("Benefit cluster-count sensitivity analysis complete")


---
# SECTION 11: Prompt sensitivity analysis (scaffold)
---

This section is a **scaffold**. It documents the prompt sensitivity analysis
that the paper Methods section claims to have run, and that the audit found
was missing in v12, v15, and (initially) v16.

To implement, copy the extraction cell (cell 14 for concerns, cell 53 for
benefits), change the `EXTRACTION_PROMPT` / `BENEFIT_EXTRACTION_PROMPT` to an
alternative variant, write the alternative outputs to a separate cache file
(otherwise the existing cache will short-circuit the new run), and compare
cluster structure via Adjusted Rand Index on the items that survive both
prompts.

Reasonable variants to test:

1. **"Issues" framing** — replace "concerns" with "issues" (broadens scope).
2. **Stricter decontextualisation** — add: "If the phrase still mentions a
   specific technology after rewriting, return NO_CONCERN instead."
3. **Looser decontextualisation** — remove the explicit decontextualisation
   instruction entirely.
4. **Phrase length** — vary "3-10 words each" to "3-15 words each" or
   "very short (3-6 words each)".

The output to compare across variants is the cluster-membership stability
(ARI) and the lens-level distinctiveness numbers. If those are stable
across variants, the headline findings are robust to prompt design.


# PART V: Export outputs

Package outputs (concerns, benefits, and comparison) for download.

---


In [ ]:
# @title Export all analysis outputs

import os
import zipfile
from pathlib import Path
from google.colab import files

# Where outputs live
OUTPUT_DIR = Path(OUTPUT_FOLDER)
ARTIFACT_DIR = Path(globals().get("ARTIFACT_DIR", "analysis_artifacts"))

ZIP_NAME = "public_dialogue_analysis_outputs.zip"
ZIP_PATH = OUTPUT_DIR / ZIP_NAME

print("Preparing ZIP archive...")

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    # Add OUTPUT_FOLDER contents
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_file() and path.name != ZIP_NAME:
            zf.write(path, arcname=f"outputs/{path.relative_to(OUTPUT_DIR)}")

    # Optionally add analysis_artifacts if it exists
    if ARTIFACT_DIR.exists():
        for path in ARTIFACT_DIR.rglob("*"):
            if path.is_file():
                zf.write(path, arcname=f"analysis_artifacts/{path.relative_to(ARTIFACT_DIR)}")

print(f"ZIP written to: {ZIP_PATH}")
print("Files included:")

for p in sorted(OUTPUT_DIR.glob("*")):
    if p.name != ZIP_NAME:
        print(" - outputs/", p.name)

if ARTIFACT_DIR.exists():
    print(" - analysis_artifacts/ (included)")

# Trigger download in Colab
files.download(str(ZIP_PATH))


---
# Appendix (Documentation): Cell reference

> **v15 note.** This appendix is inherited from v12b and describes the *structure* of the notebook, which is unchanged in v15. v15 changes the *implementation* of many cells (bug fixes, variable renames, validation gates) but not the section layout. For the full list of v15 edits see `v15_changelog.md`.

This appendix provides a one-sentence reference for what each notebook cell does (for navigation and maintenance). It is **not** part of the analytical narrative.



---

### SECTION 0: Setup and configuration

| Cell | Title | Summary |
|---|---|---|
| 0.1 | Install analysis dependencies | Installs required Python packages used throughout the notebook. |
| 0.2 | Import libraries and define configuration | Imports libraries and defines key analysis parameters used in later steps. |
| 0.3 | Define helper functions | Defines utility functions used for I/O, caching, and repeated analysis routines. |
| 0.4 | Load inputs from previous run (optional) | Optionally restores previously generated intermediate outputs to avoid re-running earlier stages. |
| 0.5 | Configure API access | Loads and validates API credentials required for model-based extraction and labelling steps. |

---

### SECTION 1: Data ingestion and preprocessing

| Cell | Title | Summary |
|---|---|---|
| 1.1 | Upload source PDF documents | Uploads the source PDF reports into the Colab runtime for processing. |
| 1.2 | Upload document metadata | Uploads and loads metadata linking each document to technology label(s) and year. |
| 1.3 | Extract paragraph-level text from PDFs | Extracts paragraph-level text chunks from PDFs and joins them to document metadata. |
| 1.4 | Summarise paragraph-level data quality | Provides high-level checks on corpus coverage and text extraction outcomes. |

---

### SECTION 2B: Benefit extraction

| Cell | Title | Summary |
|---|---|---|
| 2.1 | Extract decontextualised benefit phrases | Extracts concise benefit phrases from paragraphs while minimising explicit technology references. |
| 2.2 | Inspect sample benefit extractions | Displays sample outputs to support qualitative inspection of extraction quality. |

---

### SECTION 3B: Embedding and clustering (benefits)

| Cell | Title | Summary |
|---|---|---|
| 3.1 | Generate semantic embeddings for benefit phrases | Creates vector embeddings for each benefit phrase for downstream clustering and similarity analysis. |
| 3.2 | Cluster benefit phrase embeddings | Clusters benefit embeddings to identify recurring types of benefit in the corpus. |
| 3.3 | Characterise clusters by technology distribution | Quantifies how evenly each cluster spans technologies and classifies clusters as cross-cutting or technology-specific. |

---

### SECTION 4B: Cluster interpretation and labelling (benefits)

| Cell | Title | Summary |
|---|---|---|
| 4.1 | Extract exemplar benefit phrases per cluster | Selects representative benefits for each cluster to support human interpretation and labelling. |
| 4.2 | Assign descriptive labels to clusters | Produces short labels/descriptions for clusters to support reporting and interpretation. |

---

### SECTION 5B: Framing lens analysis (benefits)

| Cell | Title | Summary |
|---|---|---|
| 5.1 | Identify framing lenses | Proposes higher-level thematic “lenses” that group clusters into broader frames of benefit. |
| 5.2 | Map benefit clusters to framing lenses | Creates a mapping from clusters to framing lenses for aggregated analysis and visualisation. |
| 5.3 | Compute framing lens prevalence | Summarises how frequently each framing lens appears across the corpus. |

---

### SECTION 6B: Shared benefits, shared framings, and AI distinctiveness (benefits)

| Cell | Title | Summary |
|---|---|---|
| 6.1a | Shared benefits across technologies (document-weighted) | Identifies benefit clusters that appear broadly across technologies using document-level weighting. |
| 6.1b | Shared framings across technologies (document-weighted) | Identifies framing lenses that recur broadly across technologies using document-level weighting. |
| 6.2a | Compare AI and non-AI dialogues by framing lens | Compares AI dialogues to a non-AI baseline across framing lenses. |
| 6.2b | Compare AI and non-AI dialogues on cross-cutting benefits | Compares AI dialogues to a non-AI baseline using cross-cutting benefit clusters. |
| 6.2c | Visualise AI benefit fingerprint (AI vs non-AI) | Visualises the largest AI vs non-AI differences on shared benefits. |
| 6.3 | Identify distinctive AI benefits and framings | Summarises which benefits and framings most distinguish AI dialogues from the non-AI baseline. |
| 6.4 | Analyse AI-specific benefits | Examines clusters that are strongly concentrated in AI dialogues. |
| 6.5 | Visualise cross-technology benefit distribution | Visualises benefit prevalence across technologies (e.g., heatmaps) for comparative inspection. |

---

### SECTION 7B: AI benefit dynamics over time

| Cell | Title | Summary |
|---|---|---|
| 7.1 | Compute AI benefit profiles over time | Computes year-by-year AI benefit profiles to support continuous temporal analysis. |
| 7.2 | Visualise AI benefit trajectory over time | Visualises change in AI benefit profiles over time in a low-dimensional trajectory view. |
| 7.3 | Quantify stability of AI benefit profiles | Measures how stable or volatile AI benefit profiles are across years. |
| 7.4 | Analyse AI benefit salience trajectories | Tracks trajectories of benefit salience over time at cluster and/or framing-lens level. |
| 7.5 | Visualise AI benefit salience over time | Produces heatmaps/summary plots showing temporal change in AI benefit salience. |

---

### SECTION 8B: The stable core of public technology benefits

| Cell | Title | Summary |
|---|---|---|
| 8.1 | Visualise the stable core of public benefits | Identifies and visualises benefit clusters that remain consistently salient across technologies. |
| 8.2 | Visualise the benefit embedding space | Visualises the embedding space to support interpretation of the benefit landscape. |
| 8.3 | Export traceability datasets and evidence pack | Exports datasets linking clusters and results back to source paragraphs for auditability. |

---

### SECTION 9B: Robustness checks and validation (benefits)

| Cell | Title | Summary |
|---|---|---|
| 9.1 | Assess robustness of stable core and AI fingerprint | Checks whether key headline results are stable under reasonable analytic variations. |
| 9.2 | Assess volatility of AI benefit rankings | Tests how sensitive AI benefit rankings are to discursive and sampling variation over time. |
| 9.3 | Assess lexical novelty in AI benefits over time | Evaluates whether extracted AI benefits show changing vocabulary over time. |
| 9.4 | Assess novelty of benefit types in AI dialogues | Evaluates whether the types of benefits (clusters) expressed in AI dialogues change over time. |

---

### SECTION 10: Sensitivity analysis and export

| Cell | Title | Summary |
|---|---|---|
| 10.1 | Compare results across alternative cluster counts | Re-runs key headline analyses under different cluster counts to test sensitivity. |
| 10.2 | Examine robustness of data-privacy benefits over time | Provides a focused diagnostic check on a specific theme across time within AI dialogues. |
| 10.3 | Export all analysis outputs | Collects outputs (figures, tables, datasets) into an exportable folder for sharing. |
